<div dir="rtl" style="text-align:right">

# مبانی برنامه‌سازی — بخش دوم: داده‌ها و عملگرها (نسخه‌ی Jupyter Notebook)

این نوت‌بوک دو بخش دارد:

1. **نمای کامل (عین فایل HTML):** سلول بعدی خود فایل HTML اصلی را با همان رنگ‌ها، انیمیشن‌ها، منوی کناری، جست‌وجو و دکمه‌ی چاپ نمایش می‌دهد.
2. **نسخه‌ی قابل اجرا:** همان محتوا به‌صورت سلول‌های متنی و **سلول‌های کد C که تک‌تک قابل اجرا هستند**.

**طرز استفاده (Google Colab یا Jupyter):**
- اول سلول «آماده‌سازی» و سپس سلول «نمای کامل» را اجرا کنید.
- هر سلول کدی که با `%%c` شروع می‌شود با `gcc` کامپایل و اجرا می‌شود. اگر برنامه ورودی می‌خواهد، آن را جلوی `%%c` بنویسید؛ مثلاً `%%c 25\n30` یعنی دو خط ورودی «۲۵» و «۳۰».
- تکه‌کدهایی که `main` ندارند خودکار داخل `main` قرار می‌گیرند. چند نمونه‌ی «غلط‌نمایی» عمداً فقط نمایشی‌اند و اجرا نمی‌شوند.

</div>

In [ ]:
# آماده‌سازی: ثبت دستور %%c برای کامپایل و اجرای کد C
import re, subprocess, tempfile, os
from IPython.core.magic import register_cell_magic

_HEADERS = ("#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <limits.h>\n"
            "#include <float.h>\n#include <stdint.h>\n#include <stdbool.h>\n#include <math.h>\n#include <stddef.h>\n")

def run_c(code, stdin_text=""):
    """کد C را کامپایل و اجرا می‌کند. اگر main نداشته باشد، تکه‌کد را داخل main می‌گذارد."""
    if not re.search(r'\bmain\s*\(', code):
        lines = code.split("\n")
        top = [l for l in lines if l.lstrip().startswith("#")]
        body = [l for l in lines if not l.lstrip().startswith("#")]
        code = _HEADERS + "\n".join(top) + "\nint main(void) {\n" + "\n".join(body) + "\nreturn 0;\n}\n"
    with tempfile.TemporaryDirectory() as d:
        src, exe = os.path.join(d, "prog.c"), os.path.join(d, "prog")
        open(src, "w", encoding="utf-8").write(code)
        cp = subprocess.run(["gcc", "-Wall", "-Wextra", "-std=c11", src, "-o", exe, "-lm"],
                            capture_output=True, text=True)
        out = ""
        if cp.stderr.strip():
            out += "--- پیام‌های کامپایلر ---\n" + cp.stderr.replace(src, "prog.c") + "\n"
        if cp.returncode != 0:
            return out + "(کامپایل ناموفق بود)"
        try:
            rp = subprocess.run([exe], input=stdin_text, capture_output=True, text=True, timeout=5)
            out += "--- خروجی برنامه ---\n" + rp.stdout + rp.stderr
            if rp.returncode != 0:
                out += f"\n(کد خروج: {rp.returncode})"
        except subprocess.TimeoutExpired:
            out += "(اجرا بعد از ۵ ثانیه متوقف شد)"
        return out

@register_cell_magic
def c(line, cell):
    """%%c [ورودی دلخواه برنامه؛ با \\n برای خط جدید]"""
    print(run_c(cell, line.encode().decode("unicode_escape")))

<div dir="rtl" style="text-align:right">

## نمای کامل (عین فایل HTML اصلی)

</div>

In [ ]:
# نمایش کامل فایل HTML اصلی (فشرده‌شده در همین سلول) با همان ظاهر و انیمیشن‌ها
import base64, zlib, html as _html
from IPython.display import HTML, display
_PAGE_B64 = "..."
_page = zlib.decompress(base64.b64decode(_PAGE_B64)).decode('utf-8')
display(HTML('<iframe srcdoc="' + _html.escape(_page, quote=True) + '" style="width:100%;height:900px;border:1px solid #ccc;border-radius:8px;resize:vertical"></iframe>'))

<div dir="rtl" style="text-align:right">

## نسخه‌ی قابل اجرا (متن + سلول‌های کد C)

</div>

<div dir="rtl" style="text-align:right">

<div role="main">

<div id="بخش-دوم-دادهها-و-عملگرها" class="section level1">

# بخش دوم: داده‌ها و عملگرها

**مخاطب:** دانشجویان تازه‌وارد رشته کامپیوتر (بدون پیش‌نیاز خاص)

> **راهنمای استفاده برای مدرس:** این متن عمداً به تعداد مشخصی جلسه محدود نشده است. بخش‌های اصلی برای تدریس پایه طراحی شده‌اند و کادرهای «پنجره‌ی بیشتر»، «برای کنجکاوها» و «نکته‌ی مهندسی» محتوای تکمیلی‌اند؛ مدرس می‌تواند بسته به زمان، سطح کلاس و علاقه‌ی دانشجویان آن‌ها را تدریس کند یا فقط نامشان را مطرح کند تا مسیر تحقیق بعدی برای دانشجو روشن شود.

<div id="اهداف-یادگیری" class="section level2">

## اهداف یادگیری

در پایان این بخش انتظار می‌رود دانشجو بتواند:

- مفهوم **نوع داده**، **متغیر**، **مقدار** و **نمایش داده در حافظه** را از هم تفکیک کند.
- برای یک مسئله‌ی ساده نوع داده‌ی مناسب انتخاب کند و محدودیت بازه و دقت آن را توضیح دهد.
- ورودی و خروجی قالب‌بندی‌شده را با `printf` و `scanf` بنویسد و خطاهای رایج آن‌ها را تشخیص دهد.
- عملگرهای محاسباتی، رابطه‌ای، منطقی، انتسابی و بیتی را به‌درستی به‌کار ببرد.
- تفاوت `=` و `==`، تفاوت تقسیم صحیح و اعشاری، و اثر **اولویت و associativity** عملگرها را تحلیل کند.
- نمونه‌های ساده‌ی خطاهای **syntax / warning / undefined behavior / logic error** را از هم تشخیص دهد.
- در مواجهه با نتیجه‌ی عجیب، به‌جای حدس‌زدن، نوع داده، تبدیل نوع، هشدار کامپایلر و ترتیب ارزیابی را بررسی کند.

</div>

<div id="پیشنیاز-و-پسنیاز-مفهومی" class="section level2">

## پیش‌نیاز و پس‌نیاز مفهومی

**پیش‌نیاز:** آشنایی مقدماتی با ساختار یک برنامه‌ی C، مفهوم الگوریتم و چرخه‌ی کلی کامپایل/اجرا.  
**این بخش پایه‌ی مستقیم مباحث بعدی است:** شرط‌ها و حلقه‌ها، توابع، آرایه‌ها، رشته‌ها، اشاره‌گرها و مدیریت حافظه همگی بر فهم دقیق نوع داده، عملگر و حافظه تکیه دارند.

------------------------------------------------------------------------

</div>

</div>

<div id="فصل-۱-متغیرها-و-انواع-داده" class="section level1">

# فصل ۱: متغیرها و انواع داده

<div id="مقدمه" class="section level2">

## مقدمه

هر برنامه‌ای، در نهایت، کار می‌کند با **داده**. برای این‌که کامپیوتر بداند این داده‌ها را کجا و چطور نگه دارد، از **متغیر (Variable)** استفاده می‌کنیم. متغیر را می‌توان مثل یک جعبه در نظر گرفت که یک مقدار در آن نگه‌داری می‌شود و یک نام دارد که با آن به آن جعبه دسترسی پیدا می‌کنیم.

</div>

In [ ]:
%%c
int age = 20;

<div dir="rtl" style="text-align:right">

در این خط: `int` نوع داده است، `age` نام متغیر، و `20` مقدار اولیه‌ی آن.

<div id="سه-مفهوم-که-نباید-با-هم-قاطی-شوند-نام-مقدار-و-محل-ذخیره" class="section level3">

### سه مفهوم که نباید با هم قاطی شوند: نام، مقدار و محل ذخیره

در سطح مقدماتی می‌گوییم «متغیر مثل جعبه است». این تشبیه مفید است، اما از همین ابتدا بهتر است سه چیز را جدا ببینیم:

1.  **نام متغیر** (`age`) چیزی است که برنامه‌نویس در کد می‌نویسد.
2.  **نوع متغیر** (`int`) به کامپایلر می‌گوید این مقدار چگونه تفسیر شود و چه عملیات‌هایی روی آن معنادار است.
3.  **مقدار** (`20`) محتوای فعلی آن متغیر است؛ در زمان اجرا این مقدار در بخشی از حافظه نگه‌داری می‌شود.

> **پنجره‌ی بیشتر:** چند فصل بعد، با عملگر `&` می‌توان آدرس یک متغیر را دید و با «اشاره‌گر» مستقیماً درباره‌ی آدرس‌ها صحبت کرد. در این بخش فقط کافی است بدانیم متغیر علاوه بر نام و مقدار، در زمان اجرا یک محل ذخیره هم دارد.

</div>

<div id="declaration-initialization-و-assignment" class="section level3">

### Declaration، Initialization و Assignment

این سه واژه شبیه هم‌اند اما یکی نیستند:

</div>

In [ ]:
%%c
int score;        // Declaration: معرفی متغیر
int age = 18;     // Declaration + Initialization
score = 20;       // Assignment: تغییر/تعیین مقدار پس از تعریف

<div dir="rtl" style="text-align:right">

این تفکیک بعداً هنگام خواندن خطاها، کار با `const`، آرایه‌ها و اشاره‌گرها بسیار مهم می‌شود.

</div>

</div>

<div id="۱-انواع-داده-پایه" class="section level2">

## ۱. انواع داده پایه

زبان C چند نوع داده‌ی پایه دارد که هرکدام برای ذخیره‌ی نوع خاصی از اطلاعات مناسب‌اند:

<div class="table-scroll">

| نوع داده    | کاربرد                                          | اندازه‌ی معمول (بایت) | مثال                           |
|-------------|-------------------------------------------------|----------------------|--------------------------------|
| `int`       | اعداد صحیح                                      | ۴                    | `int x = 10;`                  |
| `long`      | اعداد صحیح بزرگ‌تر                               | ۴ یا ۸               | `long x = 100000L;`            |
| `long long` | اعداد صحیح خیلی بزرگ                            | ۸                    | `long long x = 10000000000LL;` |
| `float`     | اعداد اعشاری (دقت کم)                           | ۴                    | `float pi = 3.14f;`            |
| `double`    | اعداد اعشاری (دقت بالا)                         | ۸                    | `double pi = 3.14159265;`      |
| `char`      | یک کاراکتر                                      | ۱                    | `char grade = 'A';`            |
| `bool`      | مقدار درست/غلط (نیاز به `#include <stdbool.h>`) | ۱                    | `bool isValid = true;`         |

</div>

<div id="memory-layout-demo" class="memory-visual">

<div class="memory-visual-head">

<div>

**نمایش بصری اشغال حافظه**هر خانه = ۱ بایت = ۸ بیت

</div>

<div class="bit-rule">

۸ بیت → ۱ بایت

</div>

</div>

اعداد زیر اندازه‌های **معمول** روی بسیاری از سیستم‌های امروزی‌اند؛ استاندارد C برای همه‌ی آن‌ها یک اندازه‌ی ثابت را تضمین نمی‌کند. برای اندازه‌ی واقعی سیستم، `sizeof` را بررسی کنید.

<div class="memory-rows">

<div class="memory-row">

<div class="memory-label">

**char**1 بایت · 8 بیت

</div>

<div class="bytes">

<div class="byte">

<div class="byte-title">

Byte 0

</div>

<div class="bits">

</div>

</div>

</div>

</div>

<div class="memory-row">

<div class="memory-label">

**int**4 بایت · 32 بیت

</div>

<div class="bytes">

<div class="byte">

<div class="byte-title">

Byte 0

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 1

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 2

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 3

</div>

<div class="bits">

</div>

</div>

</div>

</div>

<div class="memory-row">

<div class="memory-label">

**float**4 بایت · 32 بیت

</div>

<div class="bytes">

<div class="byte">

<div class="byte-title">

Byte 0

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 1

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 2

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 3

</div>

<div class="bits">

</div>

</div>

</div>

</div>

<div class="memory-row">

<div class="memory-label">

**double**8 بایت · 64 بیت

</div>

<div class="bytes">

<div class="byte">

<div class="byte-title">

Byte 0

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 1

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 2

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 3

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 4

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 5

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 6

</div>

<div class="bits">

</div>

</div>

<div class="byte">

<div class="byte-title">

Byte 7

</div>

<div class="bits">

</div>

</div>

</div>

</div>

</div>

<div class="memory-legend">

هر مربع کوچک = ۱ بیت هر گروه ۸تایی = ۱ بایت

</div>

</div>

**نکته‌ی مهم:** اندازه‌ی دقیق این نوع‌ها به کامپایلر و سیستم‌عامل بستگی دارد. برای دیدن اندازه‌ی واقعی روی سیستم خودتان می‌توانید از عملگر `sizeof` استفاده کنید:

<div id="binary-storage-demo" class="bit-storage-lesson">

<div class="bit-lesson-head">

<div>

### یک عدد واقعاً چطور در حافظه ذخیره می‌شود؟

حافظه را می‌توان مثل ردیفی از خانه‌های کوچک دید. هر **بیت** فقط دو حالت دارد: **۰** یا **۱**. هر ۸ بیت کنار هم یک **بایت** می‌سازند.

</div>

<div class="bit-formula">

۸ بیت = ۱ بایت

</div>

</div>

<div class="bit-weight-row" dir="ltr">

ارزش بیت**1286432168421**

</div>

<div class="binary-example">

<div class="binary-label">

**5**<span class="small">ده‌دهی</span>

</div>

<div class="binary-bits" dir="ltr">

*00000101*

</div>

<div class="binary-note power-expansion" dir="ltr">

(0×2⁷) + (0×2⁶) + (0×2⁵) + (0×2⁴) + (0×2³) + (1×2²) + (0×2¹) + (1×2⁰)  
**۴ + ۱ = ۵**

</div>

</div>

<div class="binary-example">

<div class="binary-label">

**13**<span class="small">ده‌دهی</span>

</div>

<div class="binary-bits" dir="ltr">

*00001101*

</div>

<div class="binary-note power-expansion" dir="ltr">

(0×2⁷) + (0×2⁶) + (0×2⁵) + (0×2⁴) + (1×2³) + (1×2²) + (0×2¹) + (1×2⁰)  
**۸ + ۴ + ۱ = ۱۳**

</div>

</div>

<div class="binary-example">

<div class="binary-label">

**42**<span class="small">ده‌دهی</span>

</div>

<div class="binary-bits" dir="ltr">

*00101010*

</div>

<div class="binary-note power-expansion" dir="ltr">

(0×2⁷) + (0×2⁶) + (1×2⁵) + (0×2⁴) + (1×2³) + (0×2²) + (1×2¹) + (0×2⁰)  
**۳۲ + ۸ + ۲ = ۴۲**

</div>

</div>

<div class="binary-example">

<div class="binary-label">

**255**<span class="small">ده‌دهی</span>

</div>

<div class="binary-bits" dir="ltr">

*11111111*

</div>

<div class="binary-note power-expansion" dir="ltr">

(1×2⁷) + (1×2⁶) + (1×2⁵) + (1×2⁴) + (1×2³) + (1×2²) + (1×2¹) + (1×2⁰)  
**۱۲۸ + ۶۴ + ۳۲ + ۱۶ + ۸ + ۴ + ۲ + ۱ = ۲۵۵**

</div>

</div>

<div class="memory-address-demo">

<div class="address-title">

مثلاً یک `unsigned char` با مقدار ۱۳

</div>

<div class="address-cells" dir="ltr">

00001101

</div>

<div class="address-caption">

هر مربع = ۱ بیتهر ۸ مربع = ۱ بایت00001101₂ = 13₁₀

</div>

</div>

<div class="bit-tip">

**نکته‌ی مهم:** حافظه «عدد ۱۳» را به شکل نوشته‌ی «13» نگه نمی‌دارد؛ داده به الگوی بیت‌ها تبدیل می‌شود. تفسیر این بیت‌ها به نوع داده بستگی دارد.

</div>

</div>

<div id="ns-intro" class="section level3">

### مبناهای عددی: یک مقدار، چند شیوه‌ی نوشتن

تا اینجا اعداد را با رقم‌های ۰ تا ۹ نوشته‌ایم؛ یعنی در **مبنای ۱۰** (ده‌دهی). اما سخت‌افزار رایانه فقط دو حالت ۰ و ۱ می‌شناسد، و برنامه‌نویس‌ها برای خواندن راحت‌تر الگوی بیت‌ها دو مبنای کمکی هم به کار می‌برند: **هشت‌هشتی** (Octal) و **شانزده‌شانزدهی** (Hexadecimal، به‌اختصار «هگز»).

نکته‌ی کلیدی این است که **مبنا فقط شیوه‌ی نوشتن است، نه تغییر مقدار.** «سیزده» همان سیزده است، چه آن را `13` بنویسیم، چه `1101`، چه `D`؛ حافظه در هر سه حالت همان الگوی بیت‌های `00001101` را نگه می‌دارد.

<div class="ns-card">

<div class="ns-title">

یک مقدار، چهار نوشتار

</div>

همه‌ی کارت‌های زیر دقیقاً یک عدد را نشان می‌دهند.

<div class="ns-grid4">

<div class="ns-base">

<span class="ns-big">1101<sub>2</sub></span>**دودویی**<span class="small">Binary · مبنای ۲</span>

</div>

<div class="ns-base">

<span class="ns-big">15<sub>8</sub></span>**هشت‌هشتی**<span class="small">Octal · مبنای ۸</span>

</div>

<div class="ns-base">

<span class="ns-big">13<sub>10</sub></span>**ده‌دهی**<span class="small">Decimal · مبنای ۱۰</span>

</div>

<div class="ns-base">

<span class="ns-big">D<sub>16</sub></span>**شانزده‌شانزدهی**<span class="small">Hexadecimal · مبنای ۱۶</span>

</div>

</div>

<div class="ns-same-bits" aria-hidden="true">

*00001101*

</div>

الگوی بیت‌هایی که در حافظه می‌نشیند (۸ بیت = ۱ بایت)

</div>

<div class="table-scroll">

| مبنا        | ارقام مجاز              | نوشتن در C                         | چاپ با printf                                            | کاربرد متداول                    |
|-------------|-------------------------|------------------------------------|----------------------------------------------------------|----------------------------------|
| ۲ (دودویی)  | `0` و `1`               | `0b1101` (C23 و افزونه‌ی GCC/Clang) | `%b` در C23؛ پشتیبانی آن به نسخه‌ی کتابخانه‌ی C بستگی دارد | سخت‌افزار، بیت‌ها، bitmask         |
| ۸ (هشت‌هشتی) | `0` تا `7`              | `015` (پیشوند صفر)                 | `%o`                                                     | مجوزهای فایل در یونیکس           |
| ۱۰ (ده‌دهی)  | `0` تا `9`              | `13`                               | `%d`                                                     | محاسبات و ارتباط روزمره با انسان |
| ۱۶ (هگز)    | `0` تا `9` و `A` تا `F` | `0xD` (پیشوند 0x)                  | `%x` یا `%X`                                             | آدرس حافظه، رنگ‌ها، محتوای بایت‌ها |

</div>

#### چرا دقیقاً هگز و اکتال؟

دلیل اصلی این است که ۱۶ و ۸ **توان‌های دقیق عدد ۲** هستند: `16 = 2⁴` و `8 = 2³`. در نتیجه هر رقم هگز دقیقاً معادل **۴ بیت** و هر رقم هشت‌هشتی دقیقاً معادل **۳ بیت** است. این یعنی تبدیل به دودویی و برعکس، بدون هیچ محاسبه‌ای و فقط با «گروه‌بندی» انجام می‌شود (در بخش‌های بعد خواهید دید). ده‌دهی چنین خاصیتی ندارد، چون ۱۰ توانی از ۲ نیست.

- **هگز:** یک بایت (۸ بیت) دقیقاً با دو رقم هگز نوشته می‌شود؛ مثلاً `11111111` می‌شود `FF`. یک عدد ۳۲بیتی هم فقط ۸ رقم هگز می‌خواهد، در حالی که دودویی‌اش ۳۲ رقم است.
- **هشت‌هشتی:** در ماشین‌های قدیمی که طول کلمه‌شان مضرب ۳ بود (مثل ۱۲، ۲۴ یا ۳۶ بیت) بسیار طبیعی بود. امروز بیشتر در مجوزهای فایل یونیکس و ساختار `0...` در زبان C دیده می‌شود.

<div class="bit-tip">

**نکته‌ی مهم:** نوشته‌ی «10» بدون دانستن مبنا معنی ندارد: <span class="ns-num">10<sub>2</sub></span> برابر ۲ است، <span class="ns-num">10<sub>8</sub></span> برابر ۸، <span class="ns-num">10<sub>10</sub></span> برابر ۱۰ و <span class="ns-num">10<sub>16</sub></span> برابر ۱۶. به همین دلیل زبان C با پیشوندهای `0b`، `0` و `0x` مبنا را مشخص می‌کند.

</div>

</div>

<div id="ns-table16" class="section level3">

### جدول ۰ تا ۱۵ و توان‌های دو

همان‌طور که در ده‌دهی بعد از ۹ رقم‌ها تمام می‌شوند و `10` (یک «دهتایی» و صفر «یکی») می‌آید، در هگز هم بعد از `F` رقم‌ها تمام می‌شوند و `10` (که یعنی ۱۶) می‌آید. چون برای ۱۰ تا ۱۵ رقم تک‌نمادی نداریم، از حروف `A` تا `F` استفاده می‌شود.

<div class="table-scroll">

| ده‌دهی | دودویی (۴ بیتی) | هشت‌هشتی | هگز |
|-------|-----------------|---------|-----|
| `0`   | `0000`          | `0`     | `0` |
| `1`   | `0001`          | `1`     | `1` |
| `2`   | `0010`          | `2`     | `2` |
| `3`   | `0011`          | `3`     | `3` |
| `4`   | `0100`          | `4`     | `4` |
| `5`   | `0101`          | `5`     | `5` |
| `6`   | `0110`          | `6`     | `6` |
| `7`   | `0111`          | `7`     | `7` |
| `8`   | `1000`          | `10`    | `8` |
| `9`   | `1001`          | `11`    | `9` |
| `10`  | `1010`          | `12`    | `A` |
| `11`  | `1011`          | `13`    | `B` |
| `12`  | `1100`          | `14`    | `C` |
| `13`  | `1101`          | `15`    | `D` |
| `14`  | `1110`          | `16`    | `E` |
| `15`  | `1111`          | `17`    | `F` |

</div>

<div class="bit-tip">

**نکته‌ی مهم:** ستون «دودویی» را به‌خاطر بسپارید یا هر وقت لازم شد از روی ارزش بیت‌ها (`8 4 2 1`) بسازید. این جدول ۱۶ ردیفی کلید همه‌ی تبدیل‌های سریع هگز است. حروف هگز بزرگ و کوچک بودنشان فرقی ندارد: `0xff` و `0xFF` یک مقدارند.

</div>

#### توان‌های دو که بهتر است از بر باشند

<div class="table-scroll">

| n    | 2ⁿ (ده‌دهی) | هگز     | هشت‌هشتی  |
|------|------------|---------|----------|
| `0`  | `1`        | `1`     | `1`      |
| `1`  | `2`        | `2`     | `2`      |
| `2`  | `4`        | `4`     | `4`      |
| `3`  | `8`        | `8`     | `10`     |
| `4`  | `16`       | `10`    | `20`     |
| `5`  | `32`       | `20`    | `40`     |
| `6`  | `64`       | `40`    | `100`    |
| `7`  | `128`      | `80`    | `200`    |
| `8`  | `256`      | `100`   | `400`    |
| `9`  | `512`      | `200`   | `1000`   |
| `10` | `1024`     | `400`   | `2000`   |
| `11` | `2048`     | `800`   | `4000`   |
| `12` | `4096`     | `1000`  | `10000`  |
| `13` | `8192`     | `2000`  | `20000`  |
| `14` | `16384`    | `4000`  | `40000`  |
| `15` | `32768`    | `8000`  | `100000` |
| `16` | `65536`    | `10000` | `200000` |

</div>

چند عدد از این جدول در عمل بارها پیش می‌آید: `2⁸ = 256` (تعداد حالت‌های یک بایت)، `2¹⁰ = 1024` (یک «کیلو» در دنیای حافظه) و `2¹⁶ = 65536` (تعداد حالت‌های ۱۶ بیت). به همین دلیل بیشینه‌ی یک بایت بدون علامت `255 = 0xFF` و بیشینه‌ی ۱۶ بیت بدون علامت `65535 = 0xFFFF` است.

</div>

<div id="ns-to-decimal" class="section level3">

### تبدیل هر مبنا به ده‌دهی (بسط مکانی)

همه‌ی مبناها از یک قاعده‌ی واحد پیروی می‌کنند: **نمایش مکانی (Positional Notation)**. هر رقم بسته به جایگاهش در یک توان از مبنا ضرب می‌شود و جمع این حاصل‌ضرب‌ها مقدار عدد است. شمارش جایگاه‌ها از **سمت راست** و از توان صفر شروع می‌شود.

<div class="ns-formula">

value = dₖ·bᵏ + … + d₂·b² + d₁·b¹ + d₀·b⁰

</div>

در این رابطه `b` مبنا و `d` رقم‌هاست. در ده‌دهی این کار را بی‌آنکه فکر کنیم می‌کنیم: `352 = 3×100 + 5×10 + 2×1`. حالا همین را برای مبناهای دیگر انجام می‌دهیم:

<div class="ns-expand-list">

<div class="ns-expand">

<div class="ns-lhs">

<span class="ns-num">1101<sub>2</sub></span>

</div>

<div class="ns-rhs">

1×2³ + 1×2² + 0×2¹ + 1×2⁰ = 8 + 4 + 0 + 1 = **13**

</div>

</div>

<div class="ns-expand">

<div class="ns-lhs">

<span class="ns-num">755<sub>8</sub></span>

</div>

<div class="ns-rhs">

7×8² + 5×8¹ + 5×8⁰ = 448 + 40 + 5 = **493**

</div>

</div>

<div class="ns-expand">

<div class="ns-lhs">

<span class="ns-num">2F<sub>16</sub></span>

</div>

<div class="ns-rhs">

2×16¹ + 15×16⁰ = 32 + 15 = **47**  (F = 15)

</div>

</div>

<div class="ns-expand">

<div class="ns-lhs">

<span class="ns-num">CAFE<sub>16</sub></span>

</div>

<div class="ns-rhs">

12×16³ + 10×16² + 15×16¹ + 14×16⁰  
= 49152 + 2560 + 240 + 14 = **51966**

</div>

</div>

</div>

<div class="bit-tip">

**نکته‌ی مهم:** قبل از ضرب، حروف هگز را به مقدارشان تبدیل کنید (`A=10` تا `F=15`). خطای رایج این است که `F` را مثل یک رقم تک‌واحدی حساب کنند.

</div>

</div>

<div id="ns-from-decimal" class="section level3">

### تبدیل ده‌دهی به مبنای دلخواه (تقسیم‌های متوالی)

برای رفتن در جهت مخالف، عدد را **پشت‌سرهم بر مبنای مقصد تقسیم می‌کنیم** و هر بار باقیمانده را یادداشت می‌کنیم. باقیمانده‌ی هر تقسیم، «پایین‌ترین رقمِ» باقی‌مانده‌ی عدد در آن مبناست؛ پس اولین باقیمانده کم‌ارزش‌ترین رقم است و ما باید باقیمانده‌ها را **از آخر به اول** بخوانیم. تقسیم‌ها تا وقتی ادامه پیدا می‌کنند که خارج‌قسمت به صفر برسد.

مثال ثابت: تبدیل ۱۳ به دودویی.

<div class="table-scroll">

| تقسیم    | خارج‌قسمت | باقیمانده            |
|----------|----------|----------------------|
| `13 ÷ 2` | `6`      | `1` ← کم‌ارزش‌ترین رقم |
| `6 ÷ 2`  | `3`      | `0`                  |
| `3 ÷ 2`  | `1`      | `1`                  |
| `1 ÷ 2`  | `0`      | `1` ← پرارزش‌ترین رقم |

</div>

خواندن باقیمانده‌ها از پایین به بالا: <span class="ns-num">1101<sub>2</sub></span>. همین روش برای هر مبنای دیگری هم کار می‌کند؛ مثلاً `156 ÷ 16 = 9` با باقیمانده‌ی `12` که رقم هگز `C` است، و `9 ÷ 16 = 0` با باقیمانده‌ی `9`؛ پس <span class="ns-num">156<sub>10</sub> = 9C<sub>16</sub></span>. به همین ترتیب <span class="ns-num">100<sub>10</sub> = 144<sub>8</sub></span>.

<div id="ns-div" class="ns-card">

<div class="ns-title">

انیمیشن تقسیم‌های متوالی

</div>

یک عدد و یک مبنای مقصد انتخاب کنید و تقسیم‌ها را یکی‌یکی ببینید.

<div class="ns-controls">

عدد ده‌دهی (۰ تا ۱٬۰۰۰٬۰۰۰) مبنای مقصددودویی (۲)هشت‌هشتی (۸)هگز (۱۶)

پخش دوباره

</div>

<div id="ns-div-steps" class="ns-steps" aria-live="polite">

</div>

<div id="ns-div-result" class="ns-result">

</div>

</div>

</div>

<div id="ns-shortcuts" class="section level3">

### میان‌بر: دودویی ↔ هشت‌هشتی ↔ هگز

چون ۱۶ و ۸ توان‌های ۲ هستند، برای تبدیل بین دودویی، هشت‌هشتی و هگز **هیچ محاسبه‌ای لازم نیست**؛ فقط بیت‌ها را گروه‌بندی می‌کنیم.

- **دودویی ← هگز:** بیت‌ها را از **سمت راست** چهارتا‌چهارتا جدا کنید (اگر گروه آخر کم داشت، از چپ صفر اضافه کنید)، سپس هر گروه را با جدول ۰ تا ۱۵ به یک رقم هگز تبدیل کنید.
- **دودویی ← هشت‌هشتی:** همین کار، ولی گروه‌های **سه‌تایی**.
- **هگز یا هشت‌هشتی ← دودویی:** هر رقم را با ۴ (یا ۳) بیت جایگزین کنید.
- **هگز ← هشت‌هشتی (و برعکس):** از دودویی به‌عنوان واسطه استفاده کنید.

<div class="ns-card ns-static-groups">

<div class="ns-title">

مثال: <span class="ns-num">101011110011<sub>2</sub></span>

</div>

<div class="ns-groups" aria-label="گروه‌بندی چهارتایی">

<div class="ns-g">

<div class="gb">

1010

</div>

<span class="ga">↓</span><span class="gd">A</span>

</div>

<div class="ns-g">

<div class="gb">

1111

</div>

<span class="ga">↓</span><span class="gd">F</span>

</div>

<div class="ns-g">

<div class="gb">

0011

</div>

<span class="ga">↓</span><span class="gd">3</span>

</div>

</div>

گروه‌های چهارتایی ← <span class="ns-num">AF3<sub>16</sub></span>

<div class="ns-groups" aria-label="گروه‌بندی سه‌تایی">

<div class="ns-g">

<div class="gb">

101

</div>

<span class="ga">↓</span><span class="gd">5</span>

</div>

<div class="ns-g">

<div class="gb">

011

</div>

<span class="ga">↓</span><span class="gd">3</span>

</div>

<div class="ns-g">

<div class="gb">

110

</div>

<span class="ga">↓</span><span class="gd">6</span>

</div>

<div class="ns-g">

<div class="gb">

011

</div>

<span class="ga">↓</span><span class="gd">3</span>

</div>

</div>

گروه‌های سه‌تایی ← <span class="ns-num">5363<sub>8</sub></span>  (هر دو برابر ۲۸۰۳ ده‌دهی)

</div>

برعکسش هم ساده است: <span class="ns-num">3B7<sub>16</sub></span> را باز کنید: `3 → 0011`، `B → 1011`، `7 → 0111`؛ پس <span class="ns-num">001110110111<sub>2</sub></span>.

<div id="ns-lab" class="ns-card">

<div class="ns-title">

آزمایشگاه بیت‌ها

</div>

روی هر بیت کلیک کنید تا ۰ و ۱ شود، و همزمان مقدار آن در همه‌ی مبناها و گروه‌بندی‌اش را ببینید. زیر هر بیت، ارزش آن نوشته شده است.

<div class="ns-controls">

یا یک عدد ده‌دهی بنویسید (۰ تا ۶۵۵۳۵)

گروه‌بندی ۴تایی (هگز)

گروه‌بندی ۳تایی (هشت‌هشتی)

</div>

<div class="ns-modes">

13

255

1024

2803 (AF3)

51966 (CAFE)

65535

</div>

<div class="ns-bitrow">

</div>

<div class="ns-readout">

<div>

ده‌دهی

</div>

<div>

دودویی

</div>

<div>

هشت‌هشتی (در C)

</div>

<div>

هگز (در C)

</div>

</div>

<div class="ns-groups">

</div>

در حالت ۳تایی، چون ۱۶ بر ۳ بخش‌پذیر نیست، گروه اول با صفرهای اضافه‌ی سمت چپ کامل شده است.

</div>

</div>

<div id="ns-fractions" class="section level3">

### اعشار در مبناهای مختلف (برای کنجکاوها)

اعداد اعشاری هم در مبناهای دیگر قابل نوشتن‌اند. بخش صحیح مثل قبل تبدیل می‌شود؛ برای بخش اعشاری به‌جای تقسیم، **پشت‌سرهم در مبنا ضرب می‌کنیم** و هر بار بخش صحیح حاصل را یادداشت می‌کنیم (این بار از بالا به پایین خوانده می‌شود).

مثال: `0.625` در دودویی. `0.625×2 = 1.25` (رقم ۱)، `0.25×2 = 0.5` (رقم ۰)، `0.5×2 = 1.0` (رقم ۱ و پایان). نتیجه: <span class="ns-num">0.101<sub>2</sub></span>، چون `1/2 + 0/4 + 1/8 = 0.625`.

<div class="bit-tip">

**نکته‌ی مهم:** فقط کسرهایی که مخرجشان (پس از ساده‌سازی) توانی از ۲ باشد، در دودویی پایان‌پذیرند. عدد `0.1` این‌طور نیست و در دودویی به‌صورت `0.000110011001100…` تا بی‌نهایت ادامه دارد. همین دلیل ریشه‌ای [خطای نمایش اعشاری](#sec-11) در `float` و `double` است: حافظه ناچار است این رشته‌ی بی‌پایان را در تعداد محدودی بیت ببرد.

</div>

**برای کنجکاوها:** جزئیات دقیق ذخیره‌ی اعداد اعشاری در استاندارد IEEE 754 تعریف شده است و در درس‌های بعدی (و در منابع انتهای فایل) می‌توانید دنبالش کنید.

</div>

<div id="ns-real-world" class="section level3">

### هگز و اکتال در دنیای واقعی

هگز و اکتال فقط تمرین مدرسه‌ای نیستند. چند جای واقعی که حتماً با آن‌ها روبه‌رو می‌شوید:

#### ۱) رنگ‌ها در وب و طراحی

رنگ `#RRGGBB` سه بایت است: شدت قرمز، سبز و آبی، هر کدام با یک بایت (دو رقم هگز) و در بازه‌ی ۰ تا ۲۵۵. به‌عنوان نمونه، دو رنگ اصلی همین فایل را ببینید:

<div class="ns-swatch-row">

<div class="ns-swatch">

<div class="sw" style="background:#233870">

</div>

<div class="sc">

\#233870  
R = 0x23 = 35 · G = 0x38 = 56 · B = 0x70 = 112

</div>

</div>

<div class="ns-swatch">

<div class="sw" style="background:#783C64">

</div>

<div class="sc">

\#783C64  
R = 0x78 = 120 · G = 0x3C = 60 · B = 0x64 = 100

</div>

</div>

</div>

#### ۲) مجوزهای فایل در یونیکس (هشت‌هشتی)

هر فایل سه دسته مجوز دارد (مالک، گروه، بقیه) و هر دسته سه بیت: خواندن (`r`)، نوشتن (`w`)، اجرا (`x`). چون هر دسته سه بیت است، یک رقم هشت‌هشتی برایش کافی است؛ دستور `chmod 755 file` یعنی:

<div class="ns-chmod">

<div class="ns-perm">

<div class="pt">

مالک

</div>

<div class="pd">

7

</div>

<div class="pb">

*111*

</div>

<div class="pr">

rwx

</div>

</div>

<div class="ns-perm">

<div class="pt">

گروه

</div>

<div class="pd">

5

</div>

<div class="pb">

*101*

</div>

<div class="pr">

r-x

</div>

</div>

<div class="ns-perm">

<div class="pt">

بقیه

</div>

<div class="pd">

5

</div>

<div class="pb">

*101*

</div>

<div class="pr">

r-x

</div>

</div>

</div>

در مجموع `rwxr-xr-x`؛ یعنی مالک همه‌کاره است و دیگران فقط می‌توانند بخوانند و اجرا کنند.

#### ۳) کاراکترها و کد ASCII

<div class="table-scroll">

| کاراکتر | ده‌دهی | هگز    | هشت‌هشتی |
|---------|-------|--------|---------|
| `'A'`   | `65`  | `0x41` | `0101`  |
| `'a'`   | `97`  | `0x61` | `0141`  |
| `'0'`   | `48`  | `0x30` | `060`   |
| `'\n'`  | `10`  | `0x0A` | `012`   |

</div>

حروف بزرگ و کوچک انگلیسی در جدول ASCII دقیقاً `0x20` (یعنی ۳۲) با هم فاصله دارند؛ یعنی فقط یک بیت (بیت پنجم) تفاوتشان است. توضیح بیشتر در [بخش char، ASCII و Unicode](#sec-66) آمده است.

#### ۴) آدرس حافظه، شبکه و ابزارهای اشکال‌زدایی

آدرس‌ها در حافظه (مثل خروجی `%p` در `printf` یا ابزارهایی مثل GDB و Valgrind که بعداً می‌بینید)، آدرس MAC کارت شبکه (۶ بایت، مثل `00:1A:2B:3C:4D:5E`) و آدرس‌های IPv6 (۸ گروه ۱۶بیتی، هر گروه ۴ رقم هگز) همگی با هگز نوشته می‌شوند. دلیلش همان خاصیت «هر رقم = ۴ بیت» است که اندازه‌ی نوشته را کوتاه و خواندنش را ممکن می‌کند.

#### ۵) برای کنجکاوها: ترتیب بایت‌ها در حافظه (Endianness)

وقتی عددی مثل `0x12345678` بیشتر از یک بایت جا می‌گیرد، سؤال این است که کدام بایت در کم‌ترین آدرس بنشیند. روی بیشتر رایانه‌های امروزی (از جمله x86-64) روش **Little-endian** رایج است، یعنی کم‌ارزش‌ترین بایت اول می‌آید:

<div class="ns-endian">

<div class="er">

<div class="el">

نوشتار معمول عدد

</div>

<div class="eb">

12<span class="small">پرارزش‌ترین</span>345678<span class="small">کم‌ارزش‌ترین</span>

</div>

</div>

<div class="er">

<div class="el">

در حافظه (Little-endian)

</div>

<div class="eb">

78<span class="small">آدرس +0</span>56<span class="small">+1</span>34<span class="small">+2</span>12<span class="small">+3</span>

</div>

</div>

</div>

لازم نیست حالا آن را حفظ کنید؛ فقط بدانید اگر روزی بایت‌های یک عدد را در حافظه «برعکس» دیدید، خطا نیست، قاعده است.

</div>

<div id="ns-in-c" class="section level3">

### نوشتن و چاپ مبناها در C

در C مبنای یک عدد ثابت (Literal) را **پیشوند آن** مشخص می‌کند. این قاعده در بخش «ثابت‌ها» و در کتاب K&R هم آمده است؛ اما همین‌جا که در مورد مبناها حرف می‌زنیم، جمع‌بندی‌اش بهتر است:

- بدون پیشوند ← ده‌دهی: `13`
- پیشوند `0x` یا `0X` ← هگز: `0xD`
- پیشوند `0` ← هشت‌هشتی: `015`
- پیشوند `0b` ← دودویی (در C23 استاندارد شده و سال‌هاست در GCC و Clang هم به‌صورت افزونه پشتیبانی می‌شود)

</div>

In [ ]:
%%c
#include <stdio.h>

int main(void) {
    int a = 13;       // ده‌دهی
    int b = 0xD;      // هگز (پیشوند 0x)
    int c = 015;      // هشت‌هشتی (پیشوند صفر)
    int d = 0b1101;   // دودویی (C23 / افزونه‌ی GCC و Clang)

    printf("%d %d %d %d\n", a, b, c, d);   // 13 13 13 13
    return 0;
}

<div dir="rtl" style="text-align:right">

هر چهار متغیر کاملاً یک مقدار دارند؛ پیشوند فقط به کامپایلر می‌گوید رقم‌ها را چطور بخواند و روی نوع یا مقدار تأثیری ندارد.

#### چاپ یک عدد در مبناهای مختلف

برای چاپ، به‌جای `%d` از `%o`، `%x` یا `%X` استفاده می‌کنیم. اگر پیشوند هم بخواهیم، پرچم `#` آن را اضافه می‌کند (جزئیات پرچم‌ها در بخش [قالب‌بندی دقیق‌تر خروجی](#sec-34) آمده است).

</div>

In [ ]:
%%c
#include <stdio.h>

int main(void) {
    int x = 255;

    printf("%d\n", x);          // 255   (ده‌دهی)
    printf("%o\n", x);          // 377   (هشت‌هشتی)
    printf("%x\n", x);          // ff    (هگز، حروف کوچک)
    printf("%X\n", x);          // FF    (هگز، حروف بزرگ)
    printf("%#x %#o\n", x, x);  // 0xff 0377  (با پیشوند)
    printf("%04X\n", 13);       // 000D  (چهار رقم، با صفر پیشین)
    return 0;
}

<div dir="rtl" style="text-align:right">

<div class="bit-tip">

**نکته‌ی مهم:** `%x` به‌تنهایی پیشوند `0x` را چاپ نمی‌کند؛ خروجی `ff` است نه `0xff`. برای پیشوند از `%#x` یا `0x%x` استفاده کنید.

</div>

#### دام‌های رایج

</div>

In [ ]:
%%c
printf("%d\n", 10);     // 10
printf("%d\n", 010);    // 8    ← صفر ابتدایی یعنی هشت‌هشتی!
printf("%d\n", 0x10);   // 16
// int bad = 09;        // خطای کامپایل: ۹ رقم هشت‌هشتی نیست

<div dir="rtl" style="text-align:right">

همین رفتار «صفر ابتدایی = هشت‌هشتی» دلیل می‌شود برنامه‌نویس‌ها در C معمولاً عدد را با صفر پیشین نمی‌نویسند (مثلاً `007` یا `08`)، مگر واقعاً قصد هشت‌هشتی داشته باشند.

#### هگز و اکتال برای کاراکترها

</div>

In [ ]:
%%c
char c1 = 65;      // ده‌دهی
char c2 = 0x41;    // هگز
char c3 = 0101;    // هشت‌هشتی
char c4 = 'A';     // خود کاراکتر

printf("%c %c %c %c\n", c1, c2, c3, c4);   // A A A A

<div dir="rtl" style="text-align:right">

#### خواندن هگز از ورودی

همان‌طور که `%d` برای خواندن ده‌دهی است، `%x` در `scanf` هگز می‌خواند و `%o` هشت‌هشتی. متغیر مقصد در این حالت بهتر است `unsigned int` باشد (مفهوم «بدون علامت» در [بخش signed و unsigned](#sec-10) آمده است).

</div>

In [ ]:
%%c
unsigned int v;

scanf("%x", &v);          // اگر کاربر بنویسد: ff
printf("%u\n", v);        // 255

<div dir="rtl" style="text-align:right">

#### برای کنجکاوها

**چاپ دودویی بدون `%b`:** اگر کامپایلر شما `%b` را پشتیبانی نکند، می‌توان بیت‌ها را با عملگرهای بیتی (`>>` و `&`) جدا کرد. لازم نیست این کد را همین الان کامل بفهمید؛ در [بخش عملگرهای بیتی](#sec-48) دلیل کارکردش روشن می‌شود:

</div>

In [ ]:
%%c
#include <stdio.h>

int main(void) {
    unsigned char n = 13;

    printf("%d%d%d%d%d%d%d%d\n",
           (n >> 7) & 1, (n >> 6) & 1, (n >> 5) & 1, (n >> 4) & 1,
           (n >> 3) & 1, (n >> 2) & 1, (n >> 1) & 1, n & 1);   // 00001101
    return 0;
}

<div dir="rtl" style="text-align:right">

**عدد منفی در هگز:** روی سیستمی که `int` آن ۳۲ بیت است، عدد `-1` یعنی همه‌ی ۳۲ بیت برابر ۱، و در هگز می‌شود `ffffffff` (مثلاً با `printf("%x\n", (unsigned int)-1);`). این‌که چرا منفی‌ها این‌طور ذخیره می‌شوند (مکمل دو) در مبحث بخش [اعداد علامت‌دار و بدون علامت](#sec-10) و درس‌های بعدی بازتر می‌شود.

</div>

<div id="ns-mistakes" class="section level3">

### خطاهای رایج درباره‌ی مبناها

این جدول خطاهایی را جمع کرده که در کلاس‌های مقدماتی بسیار دیده می‌شود:

<div class="table-scroll">

| اشتباه رایج                                                | توضیح و راه‌حل                                                                                   |
|------------------------------------------------------------|-------------------------------------------------------------------------------------------------|
| فکر کردن به اینکه `010` و `10` یکی‌اند                      | `010` در C هشت‌هشتی است و برابر ۸. برای ده‌دهی صفر ابتدایی نگذارید.                               |
| نوشتن `08` یا `09`                                         | ارقام ۸ و ۹ در هشت‌هشتی وجود ندارند و کامپایلر خطای «invalid digit in octal constant» می‌دهد.     |
| نوشتن حرف خارج از `A` تا `F` در هگز، مثل `0xFG`            | `G` رقم هگز نیست. کامپایلر آن را به‌عنوان پسوند نامعتبر عدد می‌بیند و خطا می‌دهد.                  |
| گمان اینکه `0x10` برابر ۱۰ است                             | `0x10` یعنی `1×16 + 0 = 16`. پیشوند فقط مبنا را نشان می‌دهد و جزو مقدار نیست.                    |
| انتظار پیشوند `0x` از `%x`                                 | `%x` فقط رقم‌ها را چاپ می‌کند. برای پیشوند از `%#x` استفاده کنید.                                 |
| فکر کردن به اینکه `0xFF` و `255` دو چیز متفاوت در حافظه‌اند | هر دو یک `int` با مقدار ۲۵۵ هستند؛ حافظه به شیوه‌ی نوشتن شما کاری ندارد.                         |
| تبدیل گروه‌های دودویی بدون صفر پیشین                        | هر رقم هگز دقیقاً ۴ بیت است. مثلاً `1` باید `0001` نوشته شود، نه `1`، وگرنه گروه‌بندی به‌هم می‌ریزد. |
| خواندن عدد هگز با `%d`                                     | `scanf("%d")` ورودی `ff` را عدد نمی‌داند. برای هگز از `%x` استفاده کنید.                         |
| ریختن عددی مثل `0x100` در `unsigned char`                  | بیشینه‌ی یک بایت بدون علامت ۲۵۵ (یعنی `0xFF`) است و `0x100` (۲۵۶) سرریز می‌کند.                   |

</div>

</div>

<div id="ns-challenges" class="section level3">

### چالش‌های کوتاه با پاسخ تاشو

این چالش‌های کوتاه را قبل از باز کردن پاسخ، روی کاغذ حل کنید. هیچ‌کدام به دستور شرطی یا حلقه نیاز ندارند.

چالش ۱ — دودویی به ده‌دهی: مقدار <span class="ns-num">101101<sub>2</sub></span> چند است؟

<div class="ns-a">

`1×32 + 0×16 + 1×8 + 1×4 + 0×2 + 1×1 = 45`

</div>

چالش ۲ — ده‌دهی به هگز و دودویی: عدد ۲۰۰ را به هگز و دودویی ببرید.

<div class="ns-a">

`200 ÷ 16 = 12` با باقیمانده‌ی `8`؛ سپس `12 ÷ 16 = 0` با باقیمانده‌ی `12` که `C` است. پس <span class="ns-num">C8<sub>16</sub></span>. با جدول ۰ تا ۱۵: `C = 1100` و `8 = 1000`، پس <span class="ns-num">11001000<sub>2</sub></span>.

</div>

چالش ۳ — هگز به دودویی و ده‌دهی: مقدار `0x1F` چیست؟

<div class="ns-a">

`1 → 0001` و `F → 1111`؛ پس `00011111`. در ده‌دهی: `1×16 + 15 = 31`.

</div>

چالش ۴ — میان‌بر گروه‌بندی: <span class="ns-num">101011110011<sub>2</sub></span> را مستقیم به هگز و هشت‌هشتی ببرید.

<div class="ns-a">

چهارتایی: `1010 | 1111 | 0011 = AF3`. سه‌تایی: `101 | 011 | 110 | 011 = 5363`. پس <span class="ns-num">AF3<sub>16</sub></span> و <span class="ns-num">5363<sub>8</sub></span>.

</div>

چالش ۵ — بدون اجرا پیش‌بینی کنید: `printf("%d %d %d\n", 10, 010, 0x10);` چه چاپ می‌کند؟

<div class="ns-a">

`10 8 16`. عدد اول ده‌دهی است، دومی هشت‌هشتی (۸) و سومی هگز (۱۶).

</div>

چالش ۶ — خطایابی: کدام خط‌ها خطای کامپایل می‌دهند؟ `int a = 0xFG;` و `int b = 09;` و `int c = 0x1F;`

<div class="ns-a">

دو خط اول خطا می‌دهند. در `0xFG` حرف `G` رقم هگز نیست و در `09` رقم ۹ در هشت‌هشتی وجود ندارد. خط سوم درست است و برابر ۳۱.

</div>

چالش ۷ — چند رقم هگز لازم است تا همه‌ی عددهای ۱۶ بیتی را بنویسیم؟ بزرگ‌ترینشان چیست؟

<div class="ns-a">

هر رقم هگز ۴ بیت است، پس ۱۶ بیت یعنی ۴ رقم هگز. بزرگ‌ترین مقدار بدون علامت `0xFFFF = 65535` است.

</div>

چالش ۸ — ده‌دهی ۴۸۸۷۹ را به هگز ببرید و حدس بزنید چرا معروف است.

<div class="ns-a">

`48879 ÷ 16 = 3054` باقیمانده `15` (F)؛ `3054 ÷ 16 = 190` باقیمانده `14` (E)؛ `190 ÷ 16 = 11` باقیمانده `14` (E)؛ `11 ÷ 16 = 0` باقیمانده `11` (B). از پایین به بالا: <span class="ns-num">BEEF<sub>16</sub></span>. این کلمه‌ی هگزی که با حروف A تا F ساخته می‌شود، در دنیای برنامه‌نویسی برای نشانه‌گذاری و شوخی بسیار به کار رفته است.

</div>

</div>

<div id="ns-cheatsheet" class="section level3">

### جدول مرجع سریع و منابع این بخش

<div class="table-scroll">

| اگر می‌خواهید…              | کار کنید                                                               |
|----------------------------|------------------------------------------------------------------------|
| از مبنای b به ده‌دهی بروید  | هر رقم را در توان مناسب مبنا (از راست و از توان صفر) ضرب و جمع کنید.   |
| از ده‌دهی به مبنای b بروید  | عدد را پشت‌سرهم بر b تقسیم کنید و باقیمانده‌ها را از آخر به اول بخوانید. |
| دودویی را به هگز ببرید     | از راست ۴تا‌۴تا گروه کنید و با جدول ۰ تا ۱۵ هر گروه را رقم کنید.        |
| دودویی را به هشت‌هشتی ببرید | از راست ۳تا‌۳تا گروه کنید.                                              |
| مبنا را در C مشخص کنید     | `0b` دودویی، `0` هشت‌هشتی، `0x` هگز، بدون پیشوند ده‌دهی.                 |
| در printf چاپ کنید         | `%d` ده‌دهی، `%o` هشت‌هشتی، `%x`/`%X` هگز؛ با پرچم `#` پیشوند هم می‌آید.  |

</div>

برای مطالعه‌ی بیشتر درباره‌ی نمایش اعداد و مبناها:

- Randal E. Bryant & David R. O'Hallaron, *Computer Systems: A Programmer's Perspective (CS:APP)*، فصل ۲، بخش «Information Storage» که نماد هگز را به‌روشنی و با تمرین معرفی می‌کند.
- Brian W. Kernighan & Dennis M. Ritchie, *The C Programming Language*، بخش «Constants» برای قواعد ثابت‌های هشت‌هشتی و هگز.
- CS50x، هفته‌ی ۰ (مبنای دو و نمایش اعداد) و هفته‌ی ۴ (حافظه و هگز).
- Donald E. Knuth, *The Art of Computer Programming*، جلد ۲، بخش «Positional Number Systems» برای بحث نظری و کامل مبناها.

</div>

</div>

In [ ]:
%%c
printf("%zu\n", sizeof(int));   // معمولاً چاپ می‌کند: 4

<div dir="rtl" style="text-align:right">

**تفاوت** **`float`** **و** **`double`\*\*\*\*:** هر دو برای اعداد اعشاری هستند، اما `double` دقت بیشتری دارد (تعداد ارقام اعشار بیشتر) و در اکثر پروژه‌های واقعی ترجیح داده می‌شود، مگر جایی که حافظه بسیار محدود باشد.

<div id="long-و-long-long--وقتی-int-کافی-نیست" class="section level3">

### `long` و `long long` — وقتی `int` کافی نیست

همان‌طور که در بخش «حوزه‌ی مقادیر و سرریز» خواهیم دید، `int` معمولاً فقط تا حدود ۲.۱ میلیارد را پوشش می‌دهد. اگر بخواهیم عددی بزرگ‌تر از این (مثلاً جمعیت جهان، یا نتیجه‌ی ضرب چند عدد بزرگ) را نگه داریم، باید از `long` یا `long long` استفاده کنیم:

</div>

In [ ]:
%%c
#include <stdio.h>

int main() {
    long long population = 8000000000LL;  // برای قابل‌حمل بودن روی سیستم‌هایی که long فقط ۳۲ بیت است
    long long distance = 9460730472580800LL; // یک سال نوری بر حسب متر؛ پسوند LL یعنی «long long»

    printf("Population: %lld\n", population);
    printf("Distance (light-year): %lld\n", distance);

    return 0;
}

<div dir="rtl" style="text-align:right">

نکات کلیدی:

- برای چاپ `long` از `%ld` و برای `long long` از `%lld` استفاده کنید (نه `%d`، چون `%d` مخصوص `int` معمولی است و خروجی اشتباه می‌دهد).
- پسوند `L` یا `LL` در انتهای عدد ثابت (مثل `100000L`) به کامپایلر می‌گوید که این عدد را از همان ابتدا با نوع `long`/`long long` در نظر بگیرد، نه `int` — این کار از overflow زودهنگام در محاسبات میانی جلوگیری می‌کند.
- اندازه‌ی `long` به پیاده‌سازی وابسته است. استاندارد C برای `long long` دست‌کم بازه‌ای معادل یک عدد صحیح علامت‌دار ۶۴ بیتی را تضمین می‌کند، اما بهتر است به‌جای فرض‌کردن تعداد بایت‌ها، در صورت نیاز `sizeof` و حدود تعریف‌شده در هدرها را بررسی کنیم. برای مقادیری که از بازه‌ی معمول `int` بزرگ‌ترند، `long long` اغلب انتخاب قابل‌حمل‌تری است.

</div>

<div id="اعداد-علامتدار-و-بدون-علامت-signed--unsigned" class="section level3">

### اعداد علامت‌دار و بدون علامت (`signed` / `unsigned`)

نوع‌های صحیح در C می‌توانند علامت‌دار یا بدون علامت باشند:

</div>

In [ ]:
%%c
signed int temperature = -5;
unsigned int count = 42U;

<div dir="rtl" style="text-align:right">

`unsigned` نمی‌تواند مقدار منفی را نمایش دهد و در عوض بخش بیشتری از بیت‌ها را برای مقادیر غیرمنفی استفاده می‌کند. نکته‌ی مهم این است که ترکیب `signed` و `unsigned` در یک عبارت می‌تواند نتیجه‌های غافلگیرکننده ایجاد کند؛ پس در کد مقدماتی بهتر است بی‌دلیل از `unsigned` استفاده نکنیم.

> **برای کنجکاوها:** در `<stdint.h>` نوع‌هایی مانند `int32_t` و `uint64_t` وجود دارند که وقتی دقیقاً به تعداد مشخصی بیت نیاز داریم مفیدند. در این درس دانستن نام آن‌ها کافی است؛ جزئیاتشان در برنامه‌نویسی سیستمی اهمیت بیشتری پیدا می‌کند.

</div>

<div id="دقت-اعشاری-و-خطای-نمایش" class="section level3">

### دقت اعشاری و خطای نمایش

`float` و `double` همه‌ی اعداد اعشاری را دقیق نگه نمی‌دارند. مثلاً مقدار `0.1` در نمایش دودویی متناهی نیست و معمولاً به نزدیک‌ترین مقدار قابل‌نمایش گرد می‌شود:

</div>

In [ ]:
%%c
#include <stdio.h>

int main(void) {
    double x = 0.1 + 0.2;
    printf("%.17f\n", x);
    return 0;
}

<div dir="rtl" style="text-align:right">

ممکن است خروجی چیزی نزدیک به `0.30000000000000004` باشد. نتیجه‌ی آموزشی مهم: برای مقایسه‌ی اعداد اعشاری، همیشه `x == 0.3` بهترین انتخاب نیست.

> **پنجره‌ی بیشتر:** استاندارد رایج نمایش اعداد اعشاری در سخت‌افزار **IEEE 754** نام دارد. فعلاً فقط اسم آن را بدانید؛ در درس معماری کامپیوتر یا محاسبات عددی می‌توان عمیق‌تر به آن برگشت.

</div>

<div id="اندازهی-نوعها-معمولاً-با-تضمینشده-فرق-دارد" class="section level3">

### اندازه‌ی نوع‌ها: «معمولاً» با «تضمین‌شده» فرق دارد

استاندارد C حداقل‌هایی را تضمین می‌کند، اما الزام نمی‌کند که مثلاً `int` همیشه دقیقاً ۴ بایت باشد. برای کد قابل‌حمل:

</div>

In [ ]:
%%c
#include <stdio.h>

printf("int: %zu bytes\n", sizeof(int));
printf("double: %zu bytes\n", sizeof(double));

<div dir="rtl" style="text-align:right">

عملگر `sizeof` خروجی‌ای از نوع `size_t` می‌دهد و برای چاپ آن از `%zu` استفاده می‌کنیم.

</div>

</div>

<div id="۲-قواعد-نامگذاری-متغیرها" class="section level2">

## ۲. قواعد نام‌گذاری متغیرها

نام متغیر نمی‌تواند دلخواه باشد؛ باید از قواعد زیر پیروی کند:

- فقط می‌تواند شامل حروف، اعداد و آندرلاین (`_`) باشد.
- نمی‌تواند با عدد شروع شود (`2value` غلط است، `value2` درست است).
- به بزرگی و کوچکی حروف حساس است (`Age` و `age` دو متغیر متفاوت‌اند).
- نمی‌تواند از کلمات کلیدی زبان (مثل `int`, `for`, `return`) استفاده کند.
- بهتر است نامی معنادار و خواناکه انتخاب شود (`studentCount` بهتر از `x` است).

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
int studentCount = 30;   // خوب
int 2ndYear = 5;         // خطای کامپایل (شروع با عدد)
int student count = 10;  // خطای کامپایل (فاصله مجاز نیست)
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

**سبک‌های رایج نام‌گذاری (Naming Convention):** قواعد بالا فقط می‌گویند چه نامی «مجاز» است؛ اما برنامه‌نویس‌ها برای نام‌های چندکلمه‌ای (مثل «تعداد دانشجو») روی چند *سبک* قرارداد کرده‌اند. خود C سبک خاصی را اجباری نکرده است.

<div class="table-scroll">

| سبک                      | مثال                | منشأ و جاهایی که استفاده می‌شود                                                                                                                   |
|--------------------------|---------------------|--------------------------------------------------------------------------------------------------------------------------------------------------|
| `camelCase`              | `studentCount`      | ریشه در جامعه‌ی زبان Smalltalk (دهه‌ی ۱۹۷۰)؛ امروز برای متغیر و تابع در Java و JavaScript رایج است.                                                |
| `PascalCase`             | `StudentCount`      | به‌نام زبان Pascal (نیکلاوس ویرت، ۱۹۷۰)؛ برای نام کلاس و نوع در Java و C#، و در Windows API.                                                      |
| `snake_case`             | `student_count`     | سنتی در Unix و C؛ کتابخانه‌ی استاندارد C (مثل `size_t` و `strlen`)، هسته‌ی لینوکس، Python (استاندارد PEP 8) و Rust.                                |
| `UPPER_SNAKE_CASE`       | `MAX_STUDENTS`      | مخصوص ثابت‌ها و ماکروها در C، مثل `INT_MAX` در `<limits.h>`.                                                                                      |
| نت مجارستانی (Hungarian) | `iCount` ، `szName` | ابداع چارلز سیمونی (Charles Simonyi)، برنامه‌نویس Xerox PARC و بعد Microsoft؛ در Windows API (مثل `hWnd`) دیده می‌شود، اما امروز کمتر توصیه می‌شود. |
| `kebab-case`             | `student-count`     | در C مجاز نیست (`-` عملگر منفی است)؛ در CSS و آدرس وب‌سایت‌ها استفاده می‌شود.                                                                       |

</div>

</div>

In [ ]:
%%c
int student_count = 30;   // snake_case
int studentCount = 30;    // camelCase
#define MAX_STUDENTS 100   // UPPER_SNAKE_CASE (ثابت/ماکرو)

<div dir="rtl" style="text-align:right">

این سبک‌ها «قانون زبان» نیستند، بلکه قرارداد تیم‌ها و راهنماهای سبک کدنویسی‌اند (مثل *Linux kernel coding style*، *PEP 8* در Python و راهنمای کدنویسی Google). در درس، هر سبکی را انتخاب کردید، در کل برنامه همان را ادامه دهید.

<div id="چند-ظرافت-نامگذاری-که-ارزش-دانستن-دارند" class="section level3">

### چند ظرافت نام‌گذاری که ارزش دانستن دارند

- نام‌هایی که با دو آندرلاین (`__name`) شروع می‌شوند یا الگوهای خاصی از آندرلاین و حرف بزرگ دارند، برای پیاده‌سازی زبان/کتابخانه رزرو شده‌اند؛ برای کد دانشجویی از آن‌ها دوری کنید.
- نام متغیر **معنادار** باشد، اما لازم نیست تبدیل به یک جمله‌ی طولانی شود.
- سبک‌هایی مثل `camelCase` و `snake_case` هر دو رایج‌اند؛ مهم‌تر از انتخاب یکی، **ثبات** در کل پروژه است.
- مفهوم **Scope** در فصل توابع کامل‌تر بررسی می‌شود؛ فعلاً بدانید دو متغیر هم‌نام در محدوده‌های متفاوت الزاماً یک متغیر نیستند.

</div>

</div>

<div id="۳-حوزهی-مقادیر-و-سرریز-overflow" class="section level2">

## ۳. حوزه‌ی مقادیر و سرریز (Overflow)

هر نوع داده، به‌خاطر تعداد بیت‌های محدودی که در حافظه اشغال می‌کند، فقط می‌تواند بازه‌ی مشخصی از اعداد را نگه دارد. برای مثال `int` معمولاً بین تقریباً `-۲٬۱۴۷٬۴۸۳٬۶۴۸` تا `۲٬۱۴۷٬۴۸۳٬۶۴۷` را پوشش می‌دهد.

وقتی نتیجه‌ی یک محاسبه از بازه‌ی نوع داده خارج شود، با **سرریز (Overflow)** روبه‌رو می‌شویم. نکته‌ی علمی مهم این است که در C، رفتار **سرریز عدد صحیح علامت‌دار** (`signed int`) طبق استاندارد «تعریف‌نشده» (Undefined Behavior) است؛ بنابراین نباید روی برگشتن خودکار عدد از انتهای بازه به ابتدای آن حساب کنیم. در بسیاری از ماشین‌های امروزی ممکن است ظاهراً چنین رفتاری دیده شود، اما برنامه‌ی قابل‌اعتماد نباید به آن وابسته باشد. در مقابل، سرریز انواع `unsigned` طبق حساب پیمانه‌ای تعریف شده است.

</div>

In [ ]:
%%c
#include <limits.h>
int x = INT_MAX;      // بزرگ‌ترین مقدار ممکن برای int
printf("%d\n", x + 1); // این عبارت برای signed int وارد محدوده‌ی Undefined Behavior می‌شود

<div dir="rtl" style="text-align:right">

> این یکی از رایج‌ترین منابع باگ در برنامه‌های واقعی است — همیشه به بازه‌ی نوع داده‌ای که استفاده می‌کنید توجه کنید.

<div id="چطور-بازهی-واقعی-نوعها-را-از-خود-c-بپرسیم" class="section level3">

### چطور بازه‌ی واقعی نوع‌ها را از خود C بپرسیم؟

به‌جای حفظ‌کردن عددها، می‌توان از هدر `<limits.h>` استفاده کرد:

</div>

In [ ]:
%%c
#include <limits.h>
#include <stdio.h>

int main(void) {
    printf("INT_MIN = %d\n", INT_MIN);
    printf("INT_MAX = %d\n", INT_MAX);
    printf("UINT_MAX = %u\n", UINT_MAX);
    return 0;
}

<div dir="rtl" style="text-align:right">

برای انواع اعشاری نیز `<float.h>` ثابت‌هایی درباره‌ی دقت و محدوده ارائه می‌کند.

</div>

<div id="چهار-واژهی-خطا-که-از-همین-حالا-باید-بشناسیم" class="section level3">

### چهار واژه‌ی خطا که از همین حالا باید بشناسیم

<div class="table-scroll">

<table class="cols-3">
<colgroup>
<col style="width: 33%" />
<col style="width: 33%" />
<col style="width: 33%" />
</colgroup>
<thead>
<tr class="header">
<th>نوع مشکل</th>
<th>نمونه</th>
<th>معمولاً چه زمانی دیده می‌شود؟</th>
</tr>
</thead>
<tbody>
<tr class="odd">
<td>Syntax Error</td>
<td><ul>
<li>جا انداختن <code>;</code></li>
<li>باز یا بسته‌نشدن پرانتز و آکولاد: <code>(</code> <code>{</code></li>
<li>تمام‌نشدن رشته، یعنی جا انداختن <code>"</code> پایانی</li>
<li>نام متغیری که با رقم شروع می‌شود: <code>int 2x;</code></li>
<li>جا انداختن ویرگول بین آرگومان‌ها: <code>printf("%d" x);</code></li>
</ul></td>
<td>زمان کامپایل</td>
</tr>
<tr class="even">
<td>Warning</td>
<td><ul>
<li>تبدیل نوع مشکوک</li>
<li>متغیر تعریف‌شده ولی استفاده‌نشده (unused variable)</li>
<li>ناهماهنگی format specifier با نوع آرگومان، مثل چاپ <code>double</code> با <code>%d</code></li>
<li>خواندن متغیری که مقدار اولیه نگرفته (uninitialized)</li>
<li>جا دادن عدد بزرگ در نوع کوچک‌تر، مثل <code>char c = 300;</code></li>
</ul></td>
<td>زمان کامپایل، ولی ممکن است فایل اجرایی ساخته شود</td>
</tr>
<tr class="odd">
<td>Runtime Error</td>
<td><ul>
<li>تقسیم بر صفر صحیح در زمان اجرا</li>
<li>باقیمانده‌ی صفر: <code>x % 0</code></li>
<li>دسترسی به حافظه‌ی نامعتبر، مثل خواندن از اشاره‌گر <code>NULL</code> (Segmentation Fault)</li>
<li>بازگشت بی‌پایان تابع و پر شدن پشته (Stack Overflow)</li>
<li>شکست یک <code>assert</code> هنگام اجرا</li>
</ul></td>
<td>هنگام اجرا</td>
</tr>
<tr class="even">
<td>Logic Error</td>
<td><ul>
<li>فرمول اشتباه ولی کد قابل اجرا</li>
<li>تقسیم صحیح به‌جای اعشاری: <code>5 / 2</code> برابر <code>2</code> می‌شود، نه <code>2.5</code></li>
<li>اولویت اشتباه عملگرها: <code>a + b / 2</code> به‌جای <code>(a + b) / 2</code></li>
<li>استفاده از <code>=</code> (انتساب) به‌جای <code>==</code> (مقایسه)</li>
<li>جابه‌جا شدن نقش دو متغیر، مثل <code>سن</code> و <code>سال تولد</code> در یک فرمول</li>
</ul></td>
<td>برنامه اجرا می‌شود، خروجی غلط است</td>
</tr>
</tbody>
</table>

</div>

> **مفهوم مهم‌تر: Undefined Behavior.** بعضی عملیات در C آن‌قدر خطرناک‌اند که استاندارد نتیجه‌ی مشخصی برایشان تعریف نمی‌کند؛ مثل سرریز `signed int` یا دسترسی خارج از محدوده‌ی آرایه. این مفهوم در ادامه‌ی درس بارها برمی‌گردد.

</div>

</div>

<div id="۴-ثابتها-constants" class="section level2">

## ۴. ثابت‌ها (Constants)

گاهی می‌خواهیم مقداری در طول برنامه **هرگز تغییر نکند**. برای این کار دو روش رایج داریم:

**روش اول: کلمه‌ی کلیدی** **`const`**

</div>

In [ ]:
%%c
const float PI = 3.14159;
// PI = 3.0;  // خطای کامپایل: نمی‌توان ثابت را تغییر داد

<div dir="rtl" style="text-align:right">

**روش دوم: دستور پیش‌پردازنده** **`#define`**

</div>

In [ ]:
%%c
#define PI 3.14159

<div dir="rtl" style="text-align:right">

**تفاوت اصلی:** `const` یک متغیر واقعی با نوع مشخص است که کامپایلر آن را چک می‌کند؛ `#define` صرفاً یک جایگزینی متنی قبل از کامپایل است و نوع ندارد. در پروژه‌های مدرن، استفاده از `const` معمولاً توصیه می‌شود.

> **نکته‌ی دقیق:** `const` در C به این معناست که از طریق آن نام اجازه‌ی تغییر مقدار نداریم؛ این مفهوم با «ثابت زمان کامپایل» در همه‌ی موقعیت‌ها یکی نیست. از طرف دیگر، `#define` اصلاً متغیر ایجاد نمی‌کند و مرحله‌ی پیش‌پردازش آن را به‌صورت متنی جایگزین می‌کند. ماکروها قابلیت‌های بیشتری هم دارند، اما استفاده‌ی بی‌احتیاط از آن‌ها می‌تواند خطاهای ظریفی بسازد.

<div id="literalها-و-پسوندها" class="section level3">

### Literalها و پسوندها

خودِ عددهایی که مستقیماً در کد می‌نویسیم **literal** نام دارند. پسوندها به کامپایلر در تعیین نوع آن‌ها کمک می‌کنند:

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
42       // معمولاً int
42U      // unsigned int
42L      // long
42LL     // long long
3.14     // double
3.14f    // float
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

این موضوع زمانی مهم می‌شود که محاسبه قبل از ذخیره‌شدن در متغیر سرریز کند.

</div>

</div>

<div id="۵-تبدیل-نوع-type-casting" class="section level2">

## ۵. تبدیل نوع (Type Casting)

گاهی لازم است مقدار یک نوع داده را به نوع دیگری تبدیل کنیم.

**تبدیل ضمنی (Implicit Casting):** کامپایلر خودش، بدون این‌که از او بخواهیم، تبدیل را انجام می‌دهد.

</div>

In [ ]:
%%c
int a = 5;
double b = a;   // int به‌طور خودکار به double تبدیل می‌شود

<div dir="rtl" style="text-align:right">

**تبدیل صریح (Explicit Casting):** خودمان به‌صراحت نوع را مشخص می‌کنیم.

</div>

In [ ]:
%%c
double result = 7 / 2;         // نتیجه: 3.0 (چون هر دو عدد int هستند و تقسیم صحیح انجام می‌شود)
double correct = (double)7 / 2; // نتیجه: 3.5 (چون یکی از اعداد به double تبدیل شده)

<div dir="rtl" style="text-align:right">

> **نکته‌ی رایج برای مبتدیان:** تقسیم دو عدد `int` همیشه نتیجه‌ی `int` می‌دهد (بخش اعشاری قطع می‌شود)، حتی اگر نتیجه را در یک متغیر `double` بریزید. برای گرفتن نتیجه‌ی اعشاری، حداقل یکی از دو عدد باید `cast` شود.

**نکته‌ی کوچک — نمایش عدد در مبنای دیگر با `printf`:** همان عددی که در حافظه ذخیره شده را می‌توانید مستقیماً به‌صورت هگزادسیمال یا اکتال هم چاپ کنید، بدون هیچ تبدیل دستی:

</div>

In [ ]:
%%c
int x = 255;
printf("%d\n", x);   // 255   (ده‌دهی)
printf("%x\n", x);   // ff    (هگزادسیمال)
printf("%o\n", x);   // 377   (اکتال)

<div dir="rtl" style="text-align:right">

این ارتباط مستقیمی با مبحث سیستم‌های عددی دارد که پیش‌تر خوانده‌اید.

<div id="تبدیل-نوع-promotion-و-از-دست-رفتن-اطلاعات" class="section level3">

### تبدیل نوع، promotion و از دست رفتن اطلاعات

تبدیل نوع همیشه بی‌خطر نیست:

</div>

In [ ]:
%%c
double price = 19.99;
int whole = (int)price;   // 19؛ بخش اعشاری حذف می‌شود، گرد نمی‌شود

<div dir="rtl" style="text-align:right">

همچنین انواع کوچک‌تر مثل `char` و `short` در بسیاری از عبارت‌های محاسباتی ابتدا به `int` ارتقا پیدا می‌کنند؛ به این فرایند **integer promotion** می‌گویند.

> **پنجره‌ی بیشتر:** مجموعه قوانین تبدیل ضمنی در C با نام **usual arithmetic conversions** شناخته می‌شود. حفظ کردن جزئیات آن در این مرحله لازم نیست، اما وقتی ترکیبی از `int`، `unsigned`، `float` و `double` در یک عبارت دیدید، بدانید پشت صحنه قواعد مشخصی برای تبدیل وجود دارد.

</div>

<div id="تمرین-خطایابی-کوتاه-خروجی-را-قبل-از-اجرا-پیشبینی-کنید" class="section level3">

### تمرین خطایابی کوتاه: خروجی را قبل از اجرا پیش‌بینی کنید

</div>

In [ ]:
%%c
int a = 7;
int b = 2;
double x = a / b;
double y = (double)a / b;
printf("%.1f %.1f\n", x, y);

<div dir="rtl" style="text-align:right">

1.  خروجی `x` چیست؟
2.  خروجی `y` چیست؟
3.  تبدیل نوع در کدام مرحله باید رخ دهد تا بخش اعشاری از بین نرود؟

</div>

</div>

<div id="جمعبندی-و-تمرینهای-این-بخش" class="section level2">

## جمع‌بندی و تمرین‌های این بخش

- انواع داده: `int`, `float`, `double`, `char`, `bool`
- قواعد نام‌گذاری معتبر
- بازه‌ی مقادیر و خطر Overflow
- ثابت‌ها: `const` در برابر `#define`
- تبدیل نوع: implicit در برابر explicit

<div id="تمرین-۱--محاسبهی-سن" class="section level3">

### تمرین ۱ — محاسبه‌ی سن

برنامه‌ای بنویسید که سن کاربر را در یک `int` بگیرد، آن را در ۲ (به‌عنوان `double`) ضرب کند و با تبدیل نوع صریح چاپ کند.

</div>

<div id="تمرین-۲--رشد-باکتری-و-کشف-overflow" class="section level3">

### تمرین ۲ — رشد باکتری و کشف Overflow

فرض کنید یک جمعیت باکتری از `1` شروع می‌شود و در هر مرحله **دو برابر** می‌شود.

برنامه‌ای بنویسید که یک متغیر از نوع `int` را از `1` شروع کند و در هر مرحله مقدار آن را در `2` ضرب کند. بعد از هر مرحله مقدار را چاپ کنید و ببینید با زیاد شدن مقدار، چه اتفاقی برای عدد می‌افتد.

**هدف تمرین:** می‌خواهیم Overflow را فقط در تئوری نبینید؛ خودتان رشد سریع عدد را اجرا کنید و ببینید وقتی مقدار از ظرفیت نوع داده عبور کند چه مشکلی به وجود می‌آید.

> **راهنمایی:** فعلاً نیازی به حلقه ندارید. می‌توانید چند بار پشت سر هم بنویسید:
>
> <div dir="ltr" style="text-align:left">
>
> ```c
> > x *= 2;
> > printf("%d\n", x);
> > 
> ```
>
> </div>
>
> و مرحله‌به‌مرحله خروجی را بررسی کنید.
>
> **نکته:** رفتار Overflow برای `int` علامت‌دار در استاندارد C تضمین‌شده نیست؛ هدف این تمرین مشاهده‌ی مفهوم سرریز در عمل است. برای آزمایش کاملاً قابل‌پیش‌بینی، بعداً می‌توانید همین ایده را با انواع unsigned بررسی کنید.

</div>

<div id="تمرین-۳--مبدل-فارنهایت-به-سلسیوس" class="section level3">

### تمرین ۳ — مبدل فارنهایت به سلسیوس

برنامه‌ای بنویسید که یک دما بر حسب فارنهایت بگیرد و آن را به سلسیوس تبدیل کند (`C = (F - 32) * 5 / 9`).

> این یکی از قدیمی‌ترین و معروف‌ترین تمرین‌های آموزش C است؛ نسخه‌ی اصلی آن دقیقاً همین برنامه است که در کتاب کلاسیک ***The C Programming Language*** نوشته‌ی Brian Kernighan و Dennis Ritchie (معروف به کتاب K&R) به‌عنوان یکی از اولین برنامه‌های آموزشی کتاب معرفی شده است.

</div>

<div id="تمرین-۴--محاسبهگر-bmi" class="section level3">

### تمرین ۴ — محاسبه‌گر BMI

برنامه‌ای بنویسید که قد (بر حسب متر) و وزن (بر حسب کیلوگرم) را بگیرد و شاخص توده‌ی بدنی را حساب کند (`BMI = weight / (height * height)`).

> این تمرین یک نمونه‌ی کلاسیک برای تمرین متغیرهای اعشاری و ترتیب محاسبه است.

</div>

<div id="تمرین-۵--شکار-خطا-نوع-دادهی-نامناسب" class="section level3">

### تمرین ۵ — شکار خطا: نوع داده‌ی نامناسب

کد زیر را بررسی کنید و بدون اجرا بگویید چه اشکالاتی دارد. سپس آن را اصلاح کنید:

</div>

In [ ]:
%%c
int average = 17.8;
char grade = "A";
float pi = 3.141592653589793;
printf("%d %c %f\n", average, grade, pi);

<div dir="rtl" style="text-align:right">

هدف، تشخیص تفاوت **کاراکتر و رشته**، تبدیل ضمنی و محدودیت دقت `float` است.

</div>

<div id="تمرین-۶--مقایسهی-float-و-double" class="section level3">

### تمرین ۶ — مقایسه‌ی `float` و `double`

عدد `1.0 / 3.0` را یک‌بار در `float` و یک‌بار در `double` ذخیره کنید و هر دو را با تعداد زیادی رقم اعشار چاپ کنید. تفاوت را توضیح دهید.

</div>

<div id="تمرین-۷--بازهی-نوعها-با-limitsh" class="section level3">

### تمرین ۷ — بازه‌ی نوع‌ها با `<limits.h>`

برنامه‌ای بنویسید که `CHAR_BIT`، `INT_MIN`، `INT_MAX` و `UINT_MAX` را چاپ کند. سپس حدس بزنید سیستم شما برای `int` چند بیت استفاده می‌کند.

------------------------------------------------------------------------

</div>

</div>

</div>

<div id="فصل-۲-ورودیخروجی-و-عملگرهای-پایه" class="section level1">

# فصل ۲: ورودی/خروجی و عملگرهای پایه

<div id="۱-ورودی-و-خروجی-printf-و-scanf" class="section level2">

## ۱. ورودی و خروجی: `printf` و `scanf`

<div id="printf--چاپ-خروجی" class="section level3">

### `printf` — چاپ خروجی

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
printf("Age: %d\n", age);
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

`printf` از **Format Specifier**ها استفاده می‌کند تا بداند هر متغیر را با چه فرمتی چاپ کند:

<div class="table-scroll">

| Specifier | نوع داده                                            |
|-----------|-----------------------------------------------------|
| `%d`      | `int`                                               |
| `%f`      | `float`                                             |
| `%lf`     | `double` (در `scanf`؛ در `printf` هر دو `%f` هستند) |
| `%c`      | `char`                                              |
| `%s`      | رشته (`string` / آرایه‌ی کاراکتر)                    |

</div>

</div>

<div id="قالببندی-دقیقتر-خروجی" class="section level3">

### قالب‌بندی دقیق‌تر خروجی

`printf` فقط برای چاپ ساده نیست. می‌توان عرض، تعداد رقم اعشار و تراز را هم کنترل کرد:

</div>

In [ ]:
%%c
double gpa = 17.4567;
printf("%.2f\n", gpa);   // 17.46
printf("%8.2f\n", gpa); // حداقل عرض 8 کاراکتر

<div dir="rtl" style="text-align:right">

چند escape sequence پرکاربرد:

<div class="table-scroll">

| دنباله | معنی              |
|--------|-------------------|
| `\n`   | رفتن به خط بعد    |
| `\t`   | Tab               |
| `\"`   | چاپ `"` داخل رشته |
| `\\`   | چاپ backslash     |

</div>

> **نکته:** در `printf` برای `double` از `%f` استفاده می‌شود؛ اما در `scanf`، `%f` برای `float*` و `%lf` برای `double*` است. این تفاوت از رایج‌ترین خطاهای مبتدیان است.

</div>

<div id="scanf--گرفتن-ورودی-از-کاربر" class="section level3">

### `scanf` — گرفتن ورودی از کاربر

</div>

In [ ]:
%%c
int age;
scanf("%d", &age);

<div dir="rtl" style="text-align:right">

**نکته‌ی حیاتی:** برای متغیرهای عددی و کاراکتری، باید از علامت `&` (آدرس متغیر) قبل از نام متغیر استفاده کنید — چون `scanf` نیاز دارد بداند مقدار را **کجا** در حافظه بنویسد.

</div>

In [ ]:
%%c
int age;
float gpa;
char grade;

printf("Enter age, GPA, grade: ");
scanf("%d %f %c", &age, &gpa, &grade);

<div dir="rtl" style="text-align:right">

> برای رشته‌ها (`%s`) از `&` استفاده نمی‌شود، چون نام آرایه به‌خودی‌خود آدرس است — این نکته را در جلسات مربوط به آرایه‌ها و رشته‌ها بیشتر توضیح می‌دهیم.

</div>

<div id="مقدار-بازگشتی-scanf-را-نادیده-نگیریم" class="section level3">

### مقدار بازگشتی `scanf` را نادیده نگیریم

`scanf` تعداد ورودی‌هایی را که با موفقیت خوانده و تبدیل کرده برمی‌گرداند:

</div>

In [ ]:
%%c
int age;
int result = scanf("%d", &age);
printf("items read: %d\n", result);

<div dir="rtl" style="text-align:right">

اگر کاربر به‌جای عدد، متن وارد کند، `result` می‌تواند `0` شود. در برنامه‌های واقعی باید این حالت را مدیریت کنیم؛ بعد از یادگیری شرط‌ها می‌توانیم ورودی نامعتبر را بهتر کنترل کنیم.

</div>

<div id="فاصلهها-در-format-string" class="section level3">

### فاصله‌ها در format string

در `scanf`، وجود فاصله قبل از `%c` معمولاً عمدی است:

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
scanf(" %c", &grade);
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

این فاصله باعث می‌شود whitespace باقی‌مانده (مثلاً Enter قبلی) رد شود. بدون آن ممکن است `%c` همان `\n` باقی‌مانده را بخواند.

</div>

<div id="دربارهی-s-فقط-تا-اولین-فاصله" class="section level3">

### درباره‌ی `%s`: فقط تا اولین فاصله

`scanf("%s", name)` برای یک کلمه مناسب است، نه نام کامل چندکلمه‌ای. ضمن آن‌که باید مراقب ظرفیت آرایه بود. شکل محدودشده امن‌تر است:

</div>

In [ ]:
%%c
char name[20];
scanf("%19s", name);

<div dir="rtl" style="text-align:right">

> **پنجره‌ی بیشتر:** برای خواندن خط کامل، تابع `fgets` معمولاً انتخاب بهتری است و در مبحث رشته‌ها دوباره به آن برمی‌گردیم.

</div>

<div id="مثال-کامل-چرا-نام-آرایهی-رشته-نیاز-به--ندارد" class="section level3">

### مثال کامل: چرا نام آرایه‌ی رشته نیاز به `&` ندارد

برای این‌که این نکته کاملاً ملموس شود، بهتر است هم رشته و هم یک کاراکتر تکی را در یک برنامه با هم مقایسه کنیم و آدرس هرکدام را هم چاپ کنیم:

</div>

In [ ]:
%%c
#include <stdio.h>

int main() {
    char name[50];
    char first_letter;

    printf("Enter your full first name: ");
    // ۱. آرایه name خودش آدرس خانه اول است؛ پس & نمی‌خواهد.
    scanf("%s", name);

    printf("Enter your favorite letter: ");
    // ۲. متغیر عادی کاراکتری است؛ پس حتماً نیاز به آدرس (&) دارد.
    scanf(" %c", &first_letter);

    printf("\n--- Result ---\n");
    printf("Name: %s\n", name);
    printf("Address of name (name):     %p\n", (void*)name);
    printf("Address of name (&name[0]): %p\n", (void*)&name[0]);
    printf("Favorite letter: %c (Address: %p)\n", first_letter, (void*)&first_letter);

    return 0;
}

<div dir="rtl" style="text-align:right">

نمونه خروجی ترمینال (Terminal Output):

    Enter your full first name: Banafsheh
    Enter your favorite letter: A

    --- Result ---
    Name: Banafsheh
    Address of name (name):     0x7ff7bfe2f820
    Address of name (&name[0]): 0x7ff7bfe2f820
    Favorite letter: A (Address: 0x7ff7bfe2f853)

**نکته‌ی کلیدی از روی همین خروجی:** مقدار آدرسی که با `name` و `&name[0]` چاپ می‌شود یکسان است. بیان دقیق‌تر این است که **نام آرایه در بیشتر عبارت‌ها به اشاره‌گری به عنصر اول آرایه تبدیل می‌شود** (array-to-pointer decay). به همین دلیل `scanf("%s", name)` به اشاره‌گر خانه‌ی اول دسترسی دارد و نیازی به `&name` نیست. توجه کنید `name` و `&name` از نظر **نوع** یکسان نیستند، هرچند ممکن است هنگام چاپ آدرس مقدار عددی یکسانی نشان دهند. در مقابل، `first_letter` یک متغیر معمولی است و برای نوشتن در آن باید `&first_letter` را به `scanf` بدهیم.

</div>

</div>

<div id="۲-عملگرهای-محاسباتی-رابطهای-و-منطقی" class="section level2">

## ۲. عملگرهای محاسباتی، رابطه‌ای و منطقی

<div id="عملگرهای-محاسباتی-arithmetic" class="section level3">

### عملگرهای محاسباتی (Arithmetic)

<div class="table-scroll">

| عملگر | معنی               | مثال                        |
|-------|--------------------|-----------------------------|
| `+`   | جمع                | `5 + 2` → `7`               |
| `-`   | تفریق              | `5 - 2` → `3`               |
| `*`   | ضرب                | `5 * 2` → `10`              |
| `/`   | تقسیم              | `5 / 2` → `2` (تقسیم صحیح!) |
| `%`   | باقی‌مانده (Modulo) | `5 % 2` → `1`               |

</div>

</div>

<div id="کاربرد-جالب-رمزنگاری-سزار-caesar-cipher" class="section level3">

### کاربرد جالب: رمزنگاری سزار (Caesar Cipher)

یکی از قدیمی‌ترین روش‌های رمزنگاری در تاریخ (که به ژولیوس سزار نسبت داده می‌شود) این است: هر حرف پیام را چند جایگاه در الفبا جلو ببریم. چون هر `char` در واقع یک عدد (کد ASCII) است، این کار فقط با **جمع عددی** انجام می‌شود:

</div>

In [ ]:
%%c
#include <stdio.h>

int main() {
    char letter = 'A';
    int shift = 3;

    char encrypted = letter + shift; // 'A' (65) + 3 = 'D' (68)
    printf("Encrypted: %c\n", encrypted); // چاپ می‌کند: D

    return 0;
}

<div dir="rtl" style="text-align:right">

برای رمزگشایی، کافی است همین مقدار `shift` را از حرف رمزشده کم کنیم. نکته‌ی مهم: این نسخه‌ی ساده فقط برای یک حرف تکی و بدون عبور از انتهای الفبا (`Z` به `A`) کار می‌کند؛ نسخه‌ی کامل‌تر که همه‌ی این حالت‌ها را هم پوشش بدهد، بعد از یادگیری `if` و حلقه‌ها قابل تکمیل است.

</div>

<div id="عملگرهای-رابطهای-relational" class="section level3">

### عملگرهای رابطه‌ای (Relational)

نتیجه‌ی این عملگرها همیشه `0` (غلط) یا `1` (درست) است:

<div class="table-scroll">

| عملگر | معنی            |
|-------|-----------------|
| `==`  | برابر است با    |
| `!=`  | برابر نیست با   |
| `<`   | کوچک‌تر          |
| `>`   | بزرگ‌تر          |
| `<=`  | کوچک‌تر یا مساوی |
| `>=`  | بزرگ‌تر یا مساوی |

</div>

> **خطای رایج:** استفاده از `=` به‌جای `==` برای مقایسه. `=` عملگر انتساب است، `==` عملگر مقایسه!

</div>

<div id="عملگرهای-منطقی-logical" class="section level3">

### عملگرهای منطقی (Logical)

برای ترکیب چند شرط با هم استفاده می‌شوند:

<div class="table-scroll">

| عملگر | معنی                             | مثال                     |
|-------|----------------------------------|--------------------------|
| `&&`  | و (AND) — هر دو باید درست باشند  | `(age > 18) && (hasID)`  |
| `||`  | یا (OR) — کافی است یکی درست باشد | `(isAdmin) || (isOwner)` |
| `!`   | نقیض (NOT)                       | `!isValid`               |

</div>

</div>

<div id="افزایش-و-کاهش--و---" class="section level3">

### افزایش و کاهش: `++` و `--`

</div>

In [ ]:
%%c
int x = 5;
x++;   // x می‌شود 6
--x;   // x دوباره می‌شود 5

<div dir="rtl" style="text-align:right">

تفاوت `x++` و `++x` زمانی مهم می‌شود که داخل یک عبارت بزرگ‌تر استفاده شوند:

</div>

In [ ]:
%%c
int x = 5;
int a = x++; // a=5 سپس x=6
int b = ++x; // ابتدا x=7 سپس b=7

<div dir="rtl" style="text-align:right">

برای خوانایی، در کدهای مقدماتی بهتر است افزایش/کاهش را تا حد امکان در خط جدا بنویسیم.

</div>

<div id="ارزیابی-کوتاهمدار-short-circuit" class="section level3">

### ارزیابی کوتاه‌مدار (Short-circuit)

در عملگرهای منطقی، C همیشه لازم نیست هر دو سمت را ارزیابی کند:

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
(a != 0) && (10 / a > 2)
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

اگر `a != 0` غلط باشد، بخش دوم اجرا نمی‌شود. این رفتار **short-circuit evaluation** نام دارد و بعداً برای نوشتن شرط‌های امن بسیار مفید است.

</div>

</div>

<div id="۳-عملگرهای-انتسابی" class="section level2">

## ۳. عملگرهای انتسابی

علاوه بر `=` ساده، C راه‌های کوتاه‌تری برای «مقداردهی همراه با عملیات» دارد:

<div class="table-scroll">

| عملگر | معادل       | مثال      |
|-------|-------------|-----------|
| `+=`  | `x = x + y` | `x += 5;` |
| `-=`  | `x = x - y` | `x -= 5;` |
| `*=`  | `x = x * y` | `x *= 5;` |
| `/=`  | `x = x / y` | `x /= 5;` |
| `%=`  | `x = x % y` | `x %= 5;` |

</div>

</div>

In [ ]:
%%c
int score = 10;
score += 5;   // score اکنون برابر است با 15
score *= 2;   // score اکنون برابر است با 30

<div dir="rtl" style="text-align:right">

</div>

<div id="۴-عملگرهای-بیتی-bitwise-operators" class="section level2">

## ۴. عملگرهای بیتی (Bitwise Operators)

عملگرهای بیتی مستقیماً روی بیت‌های نمایش دودویی اعداد صحیح کار می‌کنند. این مبحث ممکن است در شروع کمی انتزاعی باشد، اما پلی مستقیم بین برنامه‌نویسی و معماری کامپیوتر است.

<div class="table-scroll">

| عملگر | نام       | ایده‌ی ساده                                     |
|-------|-----------|------------------------------------------------|
| `&`   | AND بیتی  | فقط بیت‌هایی که در هر دو عدد ۱ هستند، ۱ می‌مانند |
| `|`   | OR بیتی   | اگر حداقل یکی از دو بیت ۱ باشد، خروجی ۱ است    |
| `^`   | XOR بیتی  | اگر دو بیت متفاوت باشند، خروجی ۱ است           |
| `~`   | NOT بیتی  | هر بیت را معکوس می‌کند                          |
| `<<`  | شیفت چپ   | بیت‌ها را به چپ می‌برد                           |
| `>>`  | شیفت راست | بیت‌ها را به راست می‌برد                         |

</div>

<div id="bit-operations-demo" class="bit-operations-visual">

<div class="bit-lesson-head">

<div>

### عمل روی بیت‌ها؛ اول با چشم ببینیم

در مثال‌های زیر فقط ۴ بیت را برای ساده‌فهم شدن نشان می‌دهیم. در C، نوع واقعی عدد می‌تواند ۸، ۱۶، ۳۲ یا بیت‌های بیشتری داشته باشد.

</div>

</div>

<div class="bit-op-grid">

<div class="bit-op-card">

**AND: `&`**

<div class="bit-line">

0101**&**0011

</div>

<div class="op-result">

0001

</div>

<span class="small">فقط جاهایی ۱ می‌ماند که هر دو بیت ۱ باشند.</span>

</div>

<div class="bit-op-card">

**OR: `|`**

<div class="bit-line">

0101**\|**0011

</div>

<div class="op-result">

0111

</div>

<span class="small">اگر حداقل یکی ۱ باشد، نتیجه ۱ است.</span>

</div>

<div class="bit-op-card">

**XOR: `^`**

<div class="bit-line">

0101**^**0011

</div>

<div class="op-result">

0110

</div>

<span class="small">اگر دو بیت متفاوت باشند، نتیجه ۱ است.</span>

</div>

<div class="bit-op-card">

**NOT: `~`**

<div class="bit-line">

~0101

</div>

<div class="op-result">

1010

</div>

<span class="small">در این مثال آموزشی ۴ بیتی، هر ۰ و ۱ برعکس می‌شود.</span>

</div>

</div>

<div class="binary-math">

#### جمع و تفریق دودویی؛ بیت به بیت

<div class="math-grid">

<div class="math-card">

<div class="math-title">

جمع: ۵ + ۳ = ۸

</div>

```
  0101
+ 0011
──────
  1000
```

**قاعده‌ی پایه:** ۱+۱ یعنی ۰ و **۱ انتقالی** به ستون بعد.

</div>

<div class="math-card">

<div class="math-title">

تفریق: ۸ − ۳ = ۵

</div>

```
  1000
− 0011
──────
  0101
```

اگر در یک ستون رقم بالایی کوچک‌تر باشد، از ستون سمت چپ **قرض** می‌گیریم؛ درست شبیه تفریق ده‌دهی.

</div>

</div>

</div>

<div class="shift-visual">

**شیفت هم جابه‌جایی واقعی بیت‌هاست:**

<div class="shift-row" dir="ltr">

0011**\<\< 1**0110*۳ → ۶*

</div>

<div class="shift-row" dir="ltr">

0011**\>\> 1**0001*۳ → ۱*

</div>

</div>

<div class="bit-tip">

**ترفند ذهنی:** برای فهم عملگرهای بیتی، همیشه عدد را به بیت‌ها تبدیل کن، عملگر را ستون‌به‌ستون اجرا کن و در پایان دوباره به ده‌دهی برگردان.

</div>

</div>

مثال:

</div>

In [ ]:
%%c
unsigned int a = 5;  // 0101
unsigned int b = 3;  // 0011

printf("%u\n", a & b); // 1  -> 0001
printf("%u\n", a | b); // 7  -> 0111
printf("%u\n", a ^ b); // 6  -> 0110

<div dir="rtl" style="text-align:right">

<div id="کاربرد-ساده-زوج-یا-فرد" class="section level3">

### کاربرد ساده: زوج یا فرد

کم‌ارزش‌ترین بیت یک عدد زوج `0` و یک عدد فرد `1` است:

</div>

In [ ]:
%%c
int n = 13;
printf("%d\n", n & 1); // برای 13 خروجی 1

<div dir="rtl" style="text-align:right">

</div>

<div id="شیفت-و-توانهای-دو" class="section level3">

### شیفت و توان‌های دو

برای اعداد unsigned و در محدوده‌ی معتبر، شیفت چپ یک بیت اغلب معادل ضرب در ۲ و شیفت راست معادل تقسیم صحیح بر ۲ دیده می‌شود:

</div>

In [ ]:
%%c
unsigned int x = 7;
printf("%u\n", x << 1); // 14

<div dir="rtl" style="text-align:right">

> **هشدار علمی:** قوانین شیفت برای مقادیر علامت‌دار و شیفت‌های خارج از محدوده جزئیات مهمی دارند. برای مثال‌های آموزشی بیتی، `unsigned` مدل ذهنی امن‌تری است.

</div>

<div id="bitmask-چیست" class="section level3">

### Bitmask چیست؟

گاهی هر بیت را به‌عنوان یک «پرچم روشن/خاموش» استفاده می‌کنیم. به این الگو **bitmask** می‌گویند. این روش در مجوزهای فایل، رجیسترهای سخت‌افزاری، پروتکل‌ها و سیستم‌های نهفته بسیار رایج است. در این درس فقط آشنایی با نام و ایده کافی است.

</div>

</div>

<div id="۵-اولویت-precedence-و-جهت-وابستگی-associativity-عملگرها" class="section level2">

## ۵. اولویت (Precedence) و جهت وابستگی (Associativity) عملگرها

عبارت زیر را در نظر بگیرید:

</div>

In [ ]:
%%c
int result = 2 + 3 * 4;

<div dir="rtl" style="text-align:right">

چون `*` اولویت بیشتری از `+` دارد، نتیجه `14` است. اما هدف آموزشی مهم‌تر این است: **در کد واقعی به‌جای حفظ‌کردن جدول بزرگ اولویت‌ها، با پرانتز قصد خود را واضح کنید.**

نمونه‌ای که پرانتز خوانایی را نجات می‌دهد:

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
int ok = (age >= 18) && (hasId == 1);
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

چند نکته‌ی بسیار مهم:

- `=` اولویت پایین‌تری از بسیاری از عملگرها دارد و با `==` اشتباه نشود.
- عملگرهای منطقی `&&` و `||` از عملگرهای رابطه‌ای پایین‌ترند.
- عملگر شرطی سه‌تایی `?:` در بخش کنترل جریان معرفی می‌شود؛ دانستن نامش در این مرحله کافی است.
- **ترتیب اولویت** با **ترتیب واقعی ارزیابی operands** یک چیز نیست؛ C در بعضی عبارت‌ها ترتیب ارزیابی را تضمین نمی‌کند.

> **پنجره‌ی بیشتر:** عبارت‌هایی مثل `i = i++ + ++i;` نمونه‌های خوبی برای «کد ننویسید!» هستند. واردشدن به جزئیات sequence/ordering در این درس لازم نیست؛ پیام اصلی این است که یک متغیر را چندبار در یک عبارت پیچیده تغییر ندهید.

<div id="تمرین-تشخیص-اولویت" class="section level3">

### تمرین تشخیص اولویت

بدون اجرا، مقدار نهایی را پیش‌بینی کنید و سپس با پرانتز بازنویسی کنید:

</div>

In [ ]:
%%c
int x = 2 + 3 * 4 > 10 && 5 % 2;

<div dir="rtl" style="text-align:right">

</div>

</div>

<div id="جمعبندی-و-تمرینهای-این-بخش-1" class="section level2">

## جمع‌بندی و تمرین‌های این بخش

- `printf` برای خروجی، `scanf` برای ورودی (و اهمیت `&`)
- عملگرهای محاسباتی، رابطه‌ای، منطقی
- عملگرهای انتسابی کوتاه (`+=`, `-=`, ...)

<div id="تمرین-۱--بررسی-دو-عدد" class="section level3">

### تمرین ۱ — بررسی دو عدد

برنامه‌ای بنویسید که دو عدد از کاربر بگیرد و بررسی کند آیا هر دو مثبت هستند یا نه. در صورت مثبت بودن هر دو عدد، مجموع و باقی‌مانده‌ی تقسیم آن‌ها را چاپ کنید.

> **توجه:** در این تمرین **نباید از `if` و `else` استفاده کنید**. هدف این است که با مفاهیم و عملگرهایی که تا اینجا یاد گرفته‌اید، مسئله را حل کنید.

</div>

<div id="تمرین-۲--سیستم-بررسی-pin" class="section level3">

### تمرین ۲ — سیستم بررسی PIN

یک برنامه‌ی ساده برای ورود به یک سیستم بنویسید.

یک PIN ثابت مثل `1234` با استفاده از `const` تعریف کنید. سپس PIN را **یک بار** از کاربر بگیرید و بررسی کنید که آیا درست است یا نه. در پایان، نتیجه‌ی مناسب را چاپ کنید؛ مثلاً «ورود موفق» یا «PIN اشتباه است».

> **توجه:** در این تمرین **نباید از `if`، `else` یا حلقه استفاده کنید**. مسئله را فقط با مفاهیم و عملگرهایی که تا این بخش یاد گرفته‌اید حل کنید.

</div>

<div id="تمرین-۳--دستگاه-خودپرداز-atm" class="section level3">

### تمرین ۳ — دستگاه خودپرداز (ATM)

یک مبلغ پول به ریال (مثلاً `870000` ریال) را از کاربر بگیرید و حساب کنید به چند اسکناس `100000` ریالی و چند اسکناس `10000` ریالی نیاز است.

**هدف تمرین:** استفاده‌ی عملی از عملگرهای تقسیم (`/`)، باقی‌مانده (`%`) و عملگرهای انتسابی ترکیبی مثل `/=` و `%=`.

> **راهنمایی:** ابتدا تعداد اسکناس‌های `100000` ریالی را با تقسیم به دست آورید. سپس با استفاده از `%=` مقدار باقی‌مانده‌ی مبلغ را محاسبه کنید و از روی آن تعداد اسکناس‌های `10000` ریالی را به دست آورید.

</div>

<div id="تمرین-۴--تبدیل-ثانیه-به-ساعت-دقیقه-و-ثانیه" class="section level3">

### تمرین ۴ — تبدیل ثانیه به ساعت، دقیقه و ثانیه

عددی به‌عنوان کل ثانیه‌ها بگیرید (مثلاً `3725`) و آن را به قالب ساعت:دقیقه:ثانیه چاپ کنید. فقط با `/` و `%` قابل حل است.

</div>

<div id="تمرین-۵--تعویض-مقدار-دو-متغیر-بدون-متغیر-کمکی" class="section level3">

### تمرین ۵ — تعویض مقدار دو متغیر بدون متغیر کمکی

دو عدد از کاربر بگیرید و مقدار آن‌ها را بدون استفاده از یک متغیر سوم (کمکی) با هم جابه‌جا کنید و نتیجه را چاپ کنید.

</div>

<div id="تمرین-۶--بررسی-سال-کبیسه" class="section level3">

### تمرین ۶ — بررسی سال کبیسه

یک سال **میلادی** را از کاربر بگیرید و بررسی کنید که آیا کبیسه است یا نه. قاعده‌ی سال کبیسه در تقویم میلادی به این صورت است:

<div class="table-scroll">

| شرط                                           | نتیجه      |
|-----------------------------------------------|------------|
| بر ۴ بخش‌پذیر **نباشد**                        | کبیسه نیست |
| بر ۴ بخش‌پذیر باشد، ولی بر ۱۰۰ نه              | کبیسه است  |
| بر ۱۰۰ بخش‌پذیر باشد، ولی بر ۴۰۰ نه (مثل ۱۹۰۰) | کبیسه نیست |
| بر ۴۰۰ بخش‌پذیر باشد (مثل ۲۰۰۰)                | کبیسه است  |

</div>

نتیجه را فقط با عملگرهای `%`، `&&`، `||` و `!` (بدون `if`) به‌صورت ۰ یا ۱ چاپ کنید.

</div>

<div id="تمرین-۷--کالبدشکافی-خطاهای-ورودیخروجی" class="section level3">

### تمرین ۷ — کالبدشکافی خطاهای ورودی/خروجی

هر خط را مستقل بررسی کنید و مشخص کنید مشکل از نوع syntax، format mismatch، منطق یا محدودیت ورودی است:

</div>

In [ ]:
%%c
int x;
double y;
char c;

scanf("%f", &y);
printf("%d\n", y);
scanf("%d", x);
scanf("%c", &c);

<div dir="rtl" style="text-align:right">

سپس نسخه‌ی صحیح را بنویسید و توضیح دهید چرا اصلاح شده است.

</div>

<div id="تمرین-۸--debug-by-warning" class="section level3">

### تمرین ۸ — Debug by Warning

کد زیر را با `-Wall -Wextra` کامپایل کنید و هشدارها را بخوانید:

</div>

In [ ]:
%%c
#include <stdio.h>

int main(void) {
    int count = 10;
    double ratio = 3.7;
    printf("count=%f ratio=%d\n", count, ratio);
    return 0;
}

<div dir="rtl" style="text-align:right">

بدون اجرای برنامه، از روی warningها مشکل را پیدا و اصلاح کنید.

</div>

<div id="تمرین-۹--decode-یک-بیتماسک-ساده" class="section level3">

### تمرین ۹ — Decode یک بیت‌ماسک ساده

سه قابلیت را با سه بیت در نظر بگیرید: خواندن = بیت 0، نوشتن = بیت 1، اجرا = بیت 2. اگر مقدار مجوز `5` باشد، با `&` مشخص کنید کدام قابلیت‌ها روشن‌اند. این تمرین مقدمه‌ای کوچک برای درک مجوزهای سطح پایین و پرچم‌هاست.

</div>

<div id="تمرین-۱۰--سؤال-مفهومی-کوتاه" class="section level3">

### تمرین ۱۰ — سؤال مفهومی کوتاه

برای هر عبارت فقط بگویید «ایمن و واضح»، «قابل اجرا ولی گمراه‌کننده»، یا «دارای رفتار تعریف‌نشده/نامعتبر» و دلیل بیاورید:

</div>

<div dir="rtl" style="text-align:right">

<div dir="ltr" style="text-align:left">

```c
5 / 2
5.0 / 2
(double)(5 / 2)
(double)5 / 2
INT_MAX + 1
```

</div>

*(نمایشی — این نمونه عمداً اجرا نمی‌شود)*

</div>

<div dir="rtl" style="text-align:right">

------------------------------------------------------------------------

</div>

<div id="جدول-ارزش-منطقی-در-c" class="section level3">

### جدول ارزش منطقی در C

در C هر مقدار صحیح **صفر** در شرط، false و هر مقدار **غیرصفر** true در نظر گرفته می‌شود. عملگرهای مقایسه‌ای و منطقی معمولاً نتیجه‌ی `0` یا `1` تولید می‌کنند.

\| A \| B \| `A && B` \| `A || B` \| \|---:\|---:\|---:\|---:\| \| 0 \| 0 \| 0 \| 0 \| \| 0 \| غیرصفر \| 0 \| 1 \| \| غیرصفر \| 0 \| 0 \| 1 \| \| غیرصفر \| غیرصفر \| 1 \| 1 \|

این نکته توضیح می‌دهد چرا عبارتی مثل `if (x)` در C معنادار است؛ هرچند خود `if` در بخش کنترل جریان آموزش داده می‌شود.

</div>

<div id="char-ascii-و-unicode-یک-تفاوت-مهم" class="section level3">

### `char`، ASCII و Unicode: یک تفاوت مهم

`char` یک واحد عدد صحیح کوچک است و برای ذخیره‌ی بایت/کاراکتر به‌کار می‌رود. در مثال‌های مقدماتی معمولاً از حروف ASCII مثل `'A'` استفاده می‌کنیم؛ اما **متن فارسی را نمی‌توان با فرض «هر حرف = یک char» تحلیل کرد**. متن‌های Unicode معمولاً با رمزگذاری‌هایی مثل UTF-8 ذخیره می‌شوند و یک نویسه‌ی فارسی ممکن است چند بایت اشغال کند.

> **پنجره‌ی بیشتر:** تفاوت «character»، «byte»، «code point» و «encoding» موضوعی مهم در برنامه‌نویسی واقعی است. در این درس فقط لازم است بدانیم مدل ساده‌ی ASCII برای همه‌ی زبان‌های طبیعی کافی نیست.

</div>

<div id="چند-format-specifier-تکمیلی-که-بهتر-است-اسمشان-را-دیده-باشید" class="section level3">

### چند format specifier تکمیلی که بهتر است اسمشان را دیده باشید

<div class="table-scroll">

| قالب   | کاربرد متداول                                 |
|--------|-----------------------------------------------|
| `%u`   | `unsigned int`                                |
| `%ld`  | `long`                                        |
| `%lld` | `long long`                                   |
| `%zu`  | `size_t`، مثلاً خروجی `sizeof`                 |
| `%p`   | آدرس؛ آرگومان معمولاً به `(void*)` تبدیل می‌شود |
| `%%`   | چاپ خود علامت `%`                             |

</div>

**قالب اشتباه فقط یک مشکل ظاهری نیست.** در توابع variadic مثل `printf` و `scanf`، ناهماهنگی format specifier با نوع آرگومان می‌تواند رفتار نادرست یا حتی Undefined Behavior ایجاد کند.

</div>

</div>

<div id="جمعبندی-کلی-بخش-دوم" class="section level2">

## جمع‌بندی کلی بخش دوم

مرور نکات کلیدی کل بخش:

1.  **متغیرها و انواع داده:** `int`, `float`, `double`, `char`, `bool`، قواعد نام‌گذاری، بازه‌ی مقادیر، overflow، ثابت‌ها، تبدیل نوع
2.  **ورودی/خروجی:** `printf`/`scanf` و format specifierها
3.  **عملگرها:** محاسباتی، رابطه‌ای، منطقی، انتسابی، بیتی، و اولویت اجرای آن‌ها

**تمرین نهایی:** برنامه‌ای بنویسید که یک عدد صحیح از کاربر بگیرد و با استفاده از عملگر بیتی `&` مشخص کند زوج است یا فرد، سپس با استفاده از `<<` مقدار آن را دو برابر کرده و چاپ کند.

</div>

<div id="منابع-و-مسیر-مطالعهی-بیشتر" class="section level2">

## منابع و مسیر مطالعه‌ی بیشتر

- Brian W. Kernighan & Dennis M. Ritchie, *The C Programming Language (K&R)* — مرجع کلاسیک زبان C.
- CS50x, Week 1: C — نمونه‌ای از شیوه‌ی آموزش متغیرها، نوع‌ها، عملگرها و کامپایل در یک درس مقدماتی علوم کامپیوتر.
- cppreference: C language — مرجع فنی برای نوع‌ها، تبدیل‌ها، عملگرها و کتابخانه‌ی استاندارد.
- ISO C / compiler documentation — برای جزئیات دقیق رفتارهای تعریف‌شده، پیاده‌سازی‌وابسته و Undefined Behavior.
- CSES Problem Set (cses.fi) — مجموعه‌ی سؤال‌های برنامه‌نویسی و الگوریتمی برای تمرین بیشتر؛ در انتخاب برخی تمرین‌های این فایل، با اعمال تغییراتی، از آن کمک گرفته شده است.
- ابزارهای هوش مصنوعی مثل Claude، ChatGPT و Gemini — برای پرسیدن توضیح بیشتر، مثال و رفع ابهام؛ پاسخ‌هایشان را با منابع مرجع بالا و با اجرای واقعی کد بررسی کنید.

> **پیشنهاد به دانشجو:** لازم نیست در شروع همه‌ی این منابع را خط‌به‌خط بخوانید. مهم است بدانید برای پرسش‌های دقیق‌تر، منابع مرجع وجود دارند و بتوانید در زمان نیاز سراغشان بروید.

------------------------------------------------------------------------

</div>

<div id="تیم-تهیه-و-تدوین" class="section level2">

## تیم تهیه و تدوین

<div id="اعضای-تیم-و-تقسیم-کار" class="section level3">

### اعضای تیم و تقسیم کار

<div class="table-scroll">

| عضو تیم            | نقش و مشارکت                                                                                                                                                                                                                                                                                                                                                                                                                                |
|--------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **ملیکا محمدبیگی** | تهیه‌ی پروتوتایپ و نسخه‌ی اولیه‌ی محتوا، شامل ساختار کلی مباحث و تمرین‌های اولیه؛ همچنین طراحی اولیه‌ی فایل تمرین‌ها، و در ادامه ساخت دو فرمت دیگر از همین فایل تمرین و بررسی آن‌ها برای اطمینان از یکسان بودن با فایل HTML اصلی                                                                                                                                                                                                                   |
| **درسا عناصری**    | بازبینی علمی و ویرایشی، توسعه و بهینه‌سازی ساختار آموزشی، افزودن/حذف و بازطراحی تمرین‌ها، افزودن نکات مفهومی و خطاهای رایج، و بررسی ایده‌های آموزشی مورد استفاده در منابع و دوره‌های دانشگاهی معتبر؛ همچنین بررسی علمی فایل تمرین‌ها، سنجش سختی سؤال‌ها و ارتباط آن‌ها با یکدیگر، یکسان‌سازی و مرتب‌سازی فایل تمرین‌ها، اضافه و کم کردن چندین سؤال، و افزودن تذکرات و نکاتی برای استاد تدریس‌کننده در برخی سؤال‌ها؛ و در پایان، تهیه و تدوین فایل نهایی |
| **زینب قاراچشمه**  | بهبود و ارتقای محتوای کارگاه بر پایه‌ی نسخه‌ی اولیه‌ی ملیکا محمدبیگی که پیش‌تر توسط درسا عناصری بررسی شده بود؛ بازبینی علمی محتوا به‌صورت مشترک با درسا عناصری؛ همچنین کمک در یافتن نمونه‌سؤال‌ها و تمرین‌های خوب و مفهومی، از جمله از CSES، و اعمال تغییراتی روی آن‌ها                                                                                                                                                                              |

</div>

</div>

<div id="فرایند-تولید-و-بازبینی" class="section level3">

### فرایند تولید و بازبینی

محتوا از یک نسخه‌ی اولیه آغاز شد و سپس با تمرکز بر **صحت علمی، پیوستگی پیش‌نیاز/پس‌نیاز، تناسب با دانشجوی ترم اول، تنوع مثال‌ها، خطاهای رایج و تمرین‌های تشخیصی** توسعه یافت. در نسخه‌ی حاضر محدودیت تعداد جلسه حذف شده تا مدرس بتواند بر اساس سطح کلاس و زمان واقعی، بین «هسته‌ی اصلی» و «مطالب تکمیلی» انتخاب کند.

</div>

</div>

<div id="شفافیت-استفاده-از-هوش-مصنوعی" class="section level2">

## شفافیت درباره‌ی استفاده از هوش مصنوعی

برای شفافیت با خوانندگان این فایل، لازم است بگوییم که در تولید این مجموعه از **ابزارهای هوش مصنوعی (AI)** بهره گرفته‌ایم. این کمک در سه بخش بود:

<div class="ai-band">

<div class="ai-card">

**ایده‌گیری**گرفتن پیشنهاد برای ساختار مباحث، مثال‌ها و شیوه‌ی توضیح مفاهیم.

</div>

<div class="ai-card">

**تولید محتوا**کمک در نگارش پیش‌نویس بخش‌هایی از متن و مثال‌ها.

</div>

<div class="ai-card main">

<span class="ai-tag">بیشترین کمک</span>**تولید خود کدها و فرمت‌ها**تولید کدهای HTML و فرمت‌های Markdown و Jupyter Notebook، که بیشترین سهم کمک هوش مصنوعی در همین بخش بود.

</div>

</div>

<div class="ai-human">

با این حال، محتوای نهایی با نظارت انسانی خودِ ما، یعنی اعضای تیم تهیه و تدوین، بازبینی و تکمیل شده است.

</div>

</div>

</div>

</div>

</div>